# Setup Detectors — Swing Points, Resistance & MA Entry

Python sandbox for fine-tuning detection logic before porting to the TS backend.

**Detectors included:**
1. ATR computation
2. Fractal pivot swing highs / swing lows (simple)
3. Significant swing points (prominence + departure)
4. Resistance / support level clustering
5. Moving-average pullback entry (EMA20 & SMA50)
6. Interactive chart with all overlays

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dataclasses import dataclass, field
from typing import Literal

# If yfinance download fails with JSONDecodeError, upgrade it:
#   pip install --upgrade yfinance
# Minimum recommended version: 0.2.36+

## 1 — Fetch data

In [2]:
# Simple yfinance fetch
TICKER = "PLTR"
PERIOD = "2y"      # 1mo, 3mo, 6mo, 1y, 2y, 5y, max
INTERVAL = "1d"     # 1d, 1wk, 1h

apple = yf.Ticker(TICKER)
raw = apple.history(period=PERIOD, interval=INTERVAL)

if raw.empty:
    raise ValueError("No data returned from yfinance. Try again later or change network/VPN.")

if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

raw = raw[["Open", "High", "Low", "Close", "Volume"]].dropna().reset_index()
raw.rename(columns={"Date": "date", "Datetime": "date"}, inplace=True)

df = raw.rename(columns={
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Close": "close",
    "Volume": "volume",
})

# Keep a master copy for date-range re-runs (always refresh on new fetch)
df_all = df.copy()
DATA_TICKER = TICKER

print(df.tail())

                         date        open        high         low       close  \
495 2026-04-07 00:00:00-04:00  146.882004  150.270004  144.449997  150.070007   
496 2026-04-08 00:00:00-04:00  154.764999  156.279999  139.169998  140.759995   
497 2026-04-09 00:00:00-04:00  139.395004  139.539993  128.470001  130.490005   
498 2026-04-10 00:00:00-04:00  128.479996  129.199997  122.680000  128.059998   
499 2026-04-13 00:00:00-04:00  130.229996  134.419907  129.149994  132.369995   

        volume  
495   28671100  
496   64827700  
497   92361400  
498  116250900  
499   65288149  


## 2 — ATR & Average Bar Size

In [3]:
def true_range(df: pd.DataFrame) -> pd.Series:
    prev_close = df["close"].shift(1)
    tr1 = df["high"] - df["low"]
    tr2 = (df["high"] - prev_close).abs()
    tr3 = (df["low"] - prev_close).abs()
    return pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)


def atr_series(df: pd.DataFrame, period: int = 14) -> pd.Series:
    """Simple rolling-mean ATR (matches the TS backend)."""
    tr = true_range(df)
    return tr.rolling(window=period, min_periods=period).mean()


def average_bar_size(df: pd.DataFrame, period: int = 20) -> pd.Series:
    """SMA of (High - Low) over `period` bars."""
    return (df["high"] - df["low"]).rolling(window=period, min_periods=1).mean()


df["atr14"] = atr_series(df, 14)
df["abs20"] = average_bar_size(df, 20)
df[["close", "atr14", "abs20"]].tail(5)

,close,atr14,abs20
495,150.070007,7.137786,6.049500
496,140.759995,8.068500,6.579150
497,130.490005,8.542786,6.940700
498,128.059998,8.560643,7.021699
499,132.369995,8.272065,6.986195


## 3 — Moving Averages

In [4]:
df["ema20"] = df["close"].ewm(span=20, adjust=False).mean()
df["sma50"] = df["close"].rolling(50).mean()
df["sma200"] = df["close"].rolling(200).mean()
df[["close", "ema20", "sma50", "sma200"]].tail(5)

,close,ema20,sma50,sma200
495,150.070007,148.776547,146.1586,164.264025
496,140.759995,148.013066,145.6244,164.281325
497,130.490005,146.344203,144.9202,164.234175
498,128.059998,144.602850,144.3344,164.158325
499,132.369995,143.437817,143.9446,164.105675


## 4 — Swing Point Detectors

### 4a — Fractal Pivots (simple one-sided lookahead)

Mirrors `detectFractalPivots` in the TS backend.

In [5]:
@dataclass
class SwingPoint:
    index: int
    price: float
    type: Literal["HIGH", "LOW"]
    atr: float = 0.0
    prominence: float = 0.0


def detect_fractal_pivots(
    df: pd.DataFrame,
    lookahead: int = 10,
) -> list[SwingPoint]:
    """
    Simple one-sided lookahead fractal pivots.
    Uses ABS (average bar size) as tolerance — same as the TS version.
    """
    highs = df["high"].values
    lows = df["low"].values
    atr_vals = df["atr14"].values
    abs_vals = df["abs20"].values
    n = len(df)
    points: list[SwingPoint] = []

    for t in range(n - lookahead):
        abs_tol = abs_vals[t] if np.isfinite(abs_vals[t]) else 0
        a = atr_vals[t] if np.isfinite(atr_vals[t]) else abs_tol

        # Swing high
        is_high = all(highs[i] < highs[t] - abs_tol for i in range(t + 1, t + lookahead + 1))
        if is_high:
            points.append(SwingPoint(index=t, price=highs[t], type="HIGH", atr=a))

        # Swing low
        is_low = all(lows[i] > lows[t] + abs_tol for i in range(t + 1, t + lookahead + 1))
        if is_low:
            points.append(SwingPoint(index=t, price=lows[t], type="LOW", atr=a))

    return points


fractal_pivots = detect_fractal_pivots(df, lookahead=10)
print(f"Fractal pivots found: {len(fractal_pivots)} "
      f"({sum(1 for p in fractal_pivots if p.type == 'HIGH')} highs, "
      f"{sum(1 for p in fractal_pivots if p.type == 'LOW')} lows)")

Fractal pivots found: 29 (9 highs, 20 lows)


### 4b — Significant Swing Points (ATR-gated with prominence + departure)

Mirrors `detectSignificantSwingPoints` in the TS backend.

In [6]:
def detect_significant_swings(
    df: pd.DataFrame,
    left: int = 3,
    right: int = 3,
    atr_period: int = 14,
    prom_atr: float = 1.5,
    depart_atr: float = 2.5,
    depart_lookahead: int = 10,
    min_swing_sep: int = 7,
) -> list[SwingPoint]:
    """
    4-condition significant swing detection:
      (A) Fractal: local max/min over [t-left .. t+right]
      (B) Prominence >= prom_atr * ATR
      (C) Departure: price moves >= depart_atr * ATR after the pivot
      (D) Spacing/dedup: keep most extreme within min_swing_sep bars
    """
    highs = df["high"].values
    lows = df["low"].values
    atr_vals = atr_series(df, atr_period).values
    n = len(df)
    candidates: list[SwingPoint] = []

    for t in range(left, n - right):
        a = atr_vals[t]
        if not np.isfinite(a) or a <= 0:
            continue

        window = slice(t - left, t + right + 1)

        # --- (A) Pivot high ---
        if highs[t] == highs[window].max() and all(
            highs[i] < highs[t] for i in range(t - left, t + right + 1) if i != t
        ):
            local_min_low = lows[window].min()
            prominence = highs[t] - local_min_low

            if prominence >= prom_atr * a:
                dep_end = min(n, t + depart_lookahead + 1)
                min_low_after = lows[t + 1 : dep_end].min() if t + 1 < dep_end else np.inf
                if min_low_after <= highs[t] - depart_atr * a:
                    candidates.append(SwingPoint(t, highs[t], "HIGH", a, prominence))

        # --- (A) Pivot low ---
        if lows[t] == lows[window].min() and all(
            lows[i] > lows[t] for i in range(t - left, t + right + 1) if i != t
        ):
            local_max_high = highs[window].max()
            prominence = local_max_high - lows[t]

            if prominence >= prom_atr * a:
                dep_end = min(n, t + depart_lookahead + 1)
                max_high_after = highs[t + 1 : dep_end].max() if t + 1 < dep_end else -np.inf
                if max_high_after >= lows[t] + depart_atr * a:
                    candidates.append(SwingPoint(t, lows[t], "LOW", a, prominence))

    # --- (D) Spacing / dedup ---
    candidates.sort(key=lambda p: p.index)
    result: list[SwingPoint] = []
    for p in candidates:
        if not result:
            result.append(p)
            continue
        last = result[-1]
        if p.type == last.type and p.index - last.index <= min_swing_sep:
            keep_new = p.price > last.price if p.type == "HIGH" else p.price < last.price
            if keep_new:
                result[-1] = p
        else:
            result.append(p)

    return result


# ── Tunable parameters ──
SIG_LEFT        = 3
SIG_RIGHT       = 3
SIG_PROM_ATR    = 1.5
SIG_DEPART_ATR  = 2.5
SIG_DEPART_LOOK = 10
SIG_MIN_SEP     = 7

sig_swings = detect_significant_swings(
    df,
    left=SIG_LEFT,
    right=SIG_RIGHT,
    prom_atr=SIG_PROM_ATR,
    depart_atr=SIG_DEPART_ATR,
    depart_lookahead=SIG_DEPART_LOOK,
    min_swing_sep=SIG_MIN_SEP,
)
print(f"Significant swings: {len(sig_swings)} "
      f"({sum(1 for p in sig_swings if p.type == 'HIGH')} highs, "
      f"{sum(1 for p in sig_swings if p.type == 'LOW')} lows)")

Significant swings: 47 (21 highs, 26 lows)


## 5 — Resistance / Support Level Clustering

Groups nearby swing-point prices into horizontal zones.  
A zone that gets tested multiple times is stronger resistance/support.

In [7]:
@dataclass
class Level:
    price: float
    type: Literal["RESISTANCE", "SUPPORT"]
    touches: int
    indices: list[int] = field(default_factory=list)


def cluster_levels(
    swings: list[SwingPoint],
    merge_pct: float = 0.015,
    min_touches: int = 2,
) -> list[Level]:
    """
    Cluster swing prices within `merge_pct` of each other.
    Returns horizontal S/R levels sorted by number of touches.
    """
    if not swings:
        return []

    sorted_swings = sorted(swings, key=lambda s: s.price)
    clusters: list[list[SwingPoint]] = [[sorted_swings[0]]]

    for sp in sorted_swings[1:]:
        cluster_avg = np.mean([s.price for s in clusters[-1]])
        if abs(sp.price - cluster_avg) / cluster_avg <= merge_pct:
            clusters[-1].append(sp)
        else:
            clusters.append([sp])

    levels: list[Level] = []
    for cluster in clusters:
        if len(cluster) < min_touches:
            continue
        avg_price = np.mean([s.price for s in cluster])
        high_count = sum(1 for s in cluster if s.type == "HIGH")
        low_count = sum(1 for s in cluster if s.type == "LOW")
        level_type = "RESISTANCE" if high_count >= low_count else "SUPPORT"
        levels.append(Level(
            price=round(avg_price, 2),
            type=level_type,
            touches=len(cluster),
            indices=[s.index for s in cluster],
        ))

    levels.sort(key=lambda lv: lv.touches, reverse=True)
    return levels


# ── Tunable parameters ──
MERGE_PCT   = 0.015   # 1.5% proximity to merge prices
MIN_TOUCHES = 2       # at least 2 touches to count as a level

all_swings = fractal_pivots + sig_swings
sr_levels = cluster_levels(all_swings, merge_pct=MERGE_PCT, min_touches=MIN_TOUCHES)

print(f"S/R levels found: {len(sr_levels)}")
for lv in sr_levels:
    print(f"  {lv.type:>12s}  ${lv.price:>8.2f}  touches={lv.touches}")

S/R levels found: 18
       SUPPORT  $   21.17  touches=3
       SUPPORT  $   29.55  touches=3
       SUPPORT  $   40.76  touches=3
    RESISTANCE  $  125.36  touches=3
       SUPPORT  $  147.94  touches=3
    RESISTANCE  $  156.69  touches=3
       SUPPORT  $  161.65  touches=3
    RESISTANCE  $  188.49  touches=3
    RESISTANCE  $   25.35  touches=2
    RESISTANCE  $   26.63  touches=2
       SUPPORT  $   66.12  touches=2
    RESISTANCE  $  135.79  touches=2
       SUPPORT  $  151.06  touches=2
    RESISTANCE  $  165.71  touches=2
       SUPPORT  $  169.17  touches=2
    RESISTANCE  $  172.15  touches=2
    RESISTANCE  $  195.64  touches=2
    RESISTANCE  $  207.52  touches=2


## 6 — Moving-Average Entry Detectors

### 6a — SMA50 Pullback (Stage 2)
Mirrors `PullbackDetector` — price near SMA50 with declining volume.

### 6b — EMA20 Pullback (Post-breakout)
Mirrors `Ema20PullbackDetector` — price near EMA20 after meaningful departure.

In [8]:
@dataclass
class EntrySignal:
    index: int
    date: object
    price: float
    type: str
    stop: float
    target: float
    rr: float
    metadata: dict = field(default_factory=dict)


def detect_sma50_pullback(
    df: pd.DataFrame,
    dist_atr_max: float = 1.0,
    vol_decline_bars: int = 3,
    stop_lookback: int = 5,
    target_rr: float = 5.0,
) -> list[EntrySignal]:
    """
    SMA50 pullback: price within `dist_atr_max` ATR of SMA50,
    volume declining over last `vol_decline_bars` bars.
    """
    signals: list[EntrySignal] = []
    closes = df["close"].values
    sma50 = df["sma50"].values
    atr = df["atr14"].values
    vols = df["volume"].values
    abs_vals = df["abs20"].values

    for t in range(max(stop_lookback, vol_decline_bars), len(df)):
        if not np.isfinite(sma50[t]) or not np.isfinite(atr[t]) or atr[t] <= 0:
            continue

        # Stage-2 proxy: close > SMA50 > SMA200 (if SMA200 available)
        sma200_val = df["sma200"].values[t] if "sma200" in df.columns else np.nan
        if np.isfinite(sma200_val) and sma50[t] <= sma200_val:
            continue

        dist = abs(closes[t] - sma50[t])
        if dist > dist_atr_max * atr[t]:
            continue

        # Volume declining
        recent_vols = vols[t - vol_decline_bars + 1 : t + 1]
        if not all(recent_vols[i] > recent_vols[i + 1] for i in range(len(recent_vols) - 1)):
            continue

        abs_val = abs_vals[t] if np.isfinite(abs_vals[t]) else atr[t]
        stop = float(df["low"].values[t - stop_lookback : t + 1].min() - abs_val)
        risk = closes[t] - stop
        if risk <= 0:
            continue

        signals.append(EntrySignal(
            index=t,
            date=df["date"].iloc[t],
            price=round(closes[t], 2),
            type="SMA50_PULLBACK",
            stop=round(stop, 2),
            target=round(closes[t] + risk * target_rr, 2),
            rr=target_rr,
            metadata={"dist_atr": round(dist / atr[t], 2), "sma50": round(sma50[t], 2)},
        ))

    return signals


def detect_ema20_pullback(
    df: pd.DataFrame,
    dist_atr_max: float = 1.0,
    departure_atr: float = 2.0,
    departure_window: int = 30,
    target_rr: float = 3.0,
) -> list[EntrySignal]:
    """
    Trend-following EMA20 bull pullback:
      - EMA20 > SMA50
      - meaningful departure >= departure_atr * ATR
      - touch-based entry: low <= ema20 <= high
      - close remains reasonably near EMA20
    """
    signals: list[EntrySignal] = []
    closes = df["close"].values
    highs = df["high"].values
    lows = df["low"].values
    ema20 = df["ema20"].values
    sma50 = df["sma50"].values
    atr = df["atr14"].values

    for t in range(departure_window, len(df)):
        if not all(np.isfinite(v) for v in [ema20[t], sma50[t], atr[t]]):
            continue
        if atr[t] <= 0:
            continue

        # EMA20 > SMA50 (ordered trend)
        if ema20[t] <= sma50[t]:
            continue

        # Meaningful departure: max close in lookback >= ema20 + departure_atr * ATR
        max_close = closes[t - departure_window : t + 1].max()
        if max_close < ema20[t] + departure_atr * atr[t]:
            continue

        # Proximity guard
        dist = abs(closes[t] - ema20[t])
        if dist > dist_atr_max * atr[t]:
            continue

        touched_ema20 = lows[t] <= ema20[t] <= highs[t]
        stop = ema20[t] - 1.0 * atr[t]
        risk = closes[t] - stop
        if risk <= 0:
            continue

        signals.append(EntrySignal(
            index=t,
            date=df["date"].iloc[t],
            price=round(closes[t], 2),
            type="EMA20_TREND_FOLLOW_BULL_PULLBACK",
            stop=round(stop, 2),
            target=round(closes[t] + risk * target_rr, 2),
            rr=target_rr,
            metadata={
                "setup_class": "TREND_FOLLOWING_20EMA_BULL_PULLBACK",
                "direction": "LONG",
                "key_level": round(ema20[t], 2),
                "dist_atr": round(dist / atr[t], 2),
                "ema20": round(ema20[t], 2),
                "departure_atr": round((max_close - ema20[t]) / atr[t], 2),
                "touched_ema20": touched_ema20,
            },
        ))

    return signals


def _price_efficiency(closes: np.ndarray) -> float:
    if len(closes) < 2:
        return 1.0
    net = abs(float(closes[-1] - closes[0]))
    path = float(np.abs(np.diff(closes)).sum())
    if path <= 1e-9:
        return 1.0
    return net / path


def _recent_volume_not_expanding(vols: np.ndarray, t: int, lookback: int = 5) -> bool:
    if t < 2 * lookback:
        return True
    recent = vols[t - lookback + 1 : t + 1]
    prior = vols[t - 2 * lookback + 1 : t - lookback + 1]
    if len(recent) < lookback or len(prior) < lookback:
        return True
    avg_recent = float(np.mean(recent))
    avg_prior = float(np.mean(prior))
    if avg_prior <= 0:
        return True
    return avg_recent <= 1.1 * avg_prior


def detect_pivot_pullback_long(
    df: pd.DataFrame,
    pivot_lookback: int = 40,
    breakout_lookback: int = 15,
    breakout_buffer_atr: float = 0.25,
    dist_atr_max: float = 1.0,
    stop_lookback: int = 5,
    target_rr: float = 4.0,
) -> list[EntrySignal]:
    """
    Pivot-pullback long:
      1) Stage-2 proxy (SMA50 > SMA200)
      2) A recent breakout above a prior pivot high
      3) Current bar pulls back near/retests that pivot
      4) No obvious volume expansion during pullback
    """
    out: list[EntrySignal] = []

    closes = df["close"].values
    highs = df["high"].values
    lows = df["low"].values
    vols = df["volume"].values
    atr = df["atr14"].values
    abs_vals = df["abs20"].values
    sma50 = df["sma50"].values
    sma200 = df["sma200"].values if "sma200" in df.columns else np.full(len(df), np.nan)

    start = pivot_lookback + breakout_lookback
    for t in range(start, len(df)):
        if not all(np.isfinite(v) for v in [atr[t], sma50[t]]):
            continue
        if atr[t] <= 0:
            continue

        if np.isfinite(sma200[t]) and sma50[t] <= sma200[t]:
            continue

        p0 = t - pivot_lookback - breakout_lookback
        p1 = t - breakout_lookback
        if p1 <= p0:
            continue

        pivot_price = float(np.max(highs[p0:p1]))

        breakout_closes = closes[p1 : t + 1]
        if len(breakout_closes) == 0:
            continue

        has_breakout = bool(np.any(breakout_closes >= pivot_price + breakout_buffer_atr * atr[t]))
        if not has_breakout:
            continue

        touched_pivot = lows[t] <= pivot_price <= highs[t]
        dist = abs(closes[t] - pivot_price)
        if not touched_pivot and dist > dist_atr_max * atr[t]:
            continue

        if closes[t] < sma50[t]:
            continue

        if not _recent_volume_not_expanding(vols, t, lookback=3):
            continue

        abs_val = abs_vals[t] if np.isfinite(abs_vals[t]) and abs_vals[t] > 0 else atr[t]
        stop = float(np.min(lows[t - stop_lookback + 1 : t + 1]) - abs_val)
        risk = closes[t] - stop
        if risk <= 0:
            continue

        out.append(EntrySignal(
            index=t,
            date=df["date"].iloc[t],
            price=round(closes[t], 2),
            type="PIVOT_PULLBACK_LONG",
            stop=round(stop, 2),
            target=round(closes[t] + risk * target_rr, 2),
            rr=target_rr,
            metadata={
                "setup_class": "PIVOT_PULLBACK_LONG",
                "direction": "LONG",
                "pivot": round(pivot_price, 2),
                "touched_pivot": touched_pivot,
                "dist_atr": round(dist / atr[t], 2),
                "breakout_buffer_atr": breakout_buffer_atr,
            },
        ))

    return out


def detect_trend_long_ema20_pullback_strict(
    df: pd.DataFrame,
    sig_swings: list[SwingPoint] | None = None,
    dist_atr_max: float = 1.0,
    departure_atr: float = 2.0,
    departure_window: int = 30,
    target_rr: float = 3.0,
) -> list[EntrySignal]:
    """
    Strict trend long EMA20 pullback:
      - Ordered trend: EMA20 > SMA50 and (if available) SMA50 > SMA200
      - Meaningful departure before pullback
      - Touch EMA20 + hold above EMA20 on close
      - No expansion volume on pullback
      - Optional structure gate: recent significant swing high exists
    """
    out: list[EntrySignal] = []

    closes = df["close"].values
    highs = df["high"].values
    lows = df["low"].values
    vols = df["volume"].values
    ema20 = df["ema20"].values
    sma50 = df["sma50"].values
    sma200 = df["sma200"].values if "sma200" in df.columns else np.full(len(df), np.nan)
    atr = df["atr14"].values

    for t in range(departure_window, len(df)):
        if not all(np.isfinite(v) for v in [ema20[t], sma50[t], atr[t]]):
            continue
        if atr[t] <= 0:
            continue

        if ema20[t] <= sma50[t]:
            continue
        if np.isfinite(sma200[t]) and sma50[t] <= sma200[t]:
            continue

        max_close = float(np.max(closes[t - departure_window : t + 1]))
        if max_close < ema20[t] + departure_atr * atr[t]:
            continue

        dist = abs(closes[t] - ema20[t])
        if dist > dist_atr_max * atr[t]:
            continue

        if closes[t] < ema20[t]:
            continue

        if not _recent_volume_not_expanding(vols, t, lookback=3):
            continue

        if sig_swings is not None:
            recent_high = any(
                (s.type == "HIGH") and (t - departure_window <= s.index < t)
                for s in sig_swings
            )
            if not recent_high:
                continue

        touched_ema20 = lows[t] <= ema20[t] <= highs[t]
        stop = float(min(ema20[t] - atr[t], np.min(lows[max(0, t - 2) : t + 1]) - 0.2 * atr[t]))
        risk = closes[t] - stop
        if risk <= 0:
            continue

        out.append(EntrySignal(
            index=t,
            date=df["date"].iloc[t],
            price=round(closes[t], 2),
            type="TREND_LONG_20EMA_PULLBACK",
            stop=round(stop, 2),
            target=round(closes[t] + risk * target_rr, 2),
            rr=target_rr,
            metadata={
                "setup_class": "TREND_LONG_20EMA_PULLBACK_STRICT",
                "direction": "LONG",
                "key_level": round(ema20[t], 2),
                "ema20": round(ema20[t], 2),
                "sma50": round(sma50[t], 2),
                "dist_atr": round(dist / atr[t], 2),
                "departure_atr": round((max_close - ema20[t]) / atr[t], 2),
                "touched_ema20": touched_ema20,
            },
        ))

    return out


def detect_base_failure_rally_short(
    df: pd.DataFrame,
    dist_atr_max: float = 1.0,
    failure_lookback: int = 20,
    target_rr: float = 3.0,
) -> list[EntrySignal]:
    """
    Base-failure rally short (first rally-back style):
      - Prior failure: close broke below SMA50 recently
      - Weak rally into SMA50 or EMA20 resistance
      - Prefer EMA20 < SMA50 structure, close remains below resistance MA
      - Weakness filter: low price efficiency and/or non-expanding volume
    """
    out: list[EntrySignal] = []

    closes = df["close"].values
    highs = df["high"].values
    vols = df["volume"].values
    atr = df["atr14"].values
    ema20 = df["ema20"].values
    sma50 = df["sma50"].values

    for t in range(max(12, failure_lookback), len(df)):
        if not all(np.isfinite(v) for v in [atr[t], ema20[t], sma50[t]]):
            continue
        if atr[t] <= 0:
            continue

        look = slice(t - failure_lookback, t)
        broke_below_50 = bool(np.any(closes[look] < (sma50[look] - 0.25 * atr[look])))
        if not broke_below_50:
            continue

        if ema20[t] >= sma50[t]:
            continue

        recent_eff = _price_efficiency(closes[t - 10 : t + 1])
        weak_eff = recent_eff < 0.35
        weak_vol = _recent_volume_not_expanding(vols, t, lookback=5)
        if not (weak_eff or weak_vol):
            continue

        dist_sma50 = abs(highs[t] - sma50[t])
        dist_ema20 = abs(highs[t] - ema20[t])

        variant = None
        pivot = None
        if dist_sma50 <= dist_atr_max * atr[t] and closes[t] < sma50[t]:
            variant = "SMA50"
            pivot = float(sma50[t])
        elif dist_ema20 <= dist_atr_max * atr[t] and closes[t] < ema20[t]:
            ema20_slope_neg = ema20[t] < ema20[max(0, t - 5)]
            if not ema20_slope_neg:
                continue
            variant = "EMA20"
            pivot = float(ema20[t])

        if variant is None or pivot is None:
            continue

        stop = pivot + 0.5 * atr[t]
        risk = stop - closes[t]
        if risk <= 0:
            continue

        out.append(EntrySignal(
            index=t,
            date=df["date"].iloc[t],
            price=round(closes[t], 2),
            type="BASE_FAILURE_SHORT",
            stop=round(stop, 2),
            target=round(closes[t] - risk * target_rr, 2),
            rr=target_rr,
            metadata={
                "setup_class": "BASE_FAILURE_RALLY_SHORT",
                "direction": "SHORT",
                "key_level": round(pivot, 2),
                "ma_resistance": variant,
                "ema20": round(ema20[t], 2),
                "sma50": round(sma50[t], 2),
                "price_efficiency_10": round(recent_eff, 3),
                "weak_volume": weak_vol,
            },
        ))

    return out


def detect_trend_short_ema20_rally(
    df: pd.DataFrame,
    sig_swings: list[SwingPoint] | None = None,
    dist_atr_max: float = 1.0,
    departure_atr: float = 2.0,
    departure_window: int = 30,
    target_rr: float = 3.0,
) -> list[EntrySignal]:
    """
    Trend short EMA20 rally (mirror of long strict):
      - Ordered downtrend: EMA20 < SMA50
      - Meaningful downside departure from EMA20
      - Rally touches EMA20 then rejects below it
      - Distance and volume filters to reduce noise
    """
    out: list[EntrySignal] = []

    closes = df["close"].values
    highs = df["high"].values
    lows = df["low"].values
    vols = df["volume"].values
    ema20 = df["ema20"].values
    sma50 = df["sma50"].values
    sma200 = df["sma200"].values if "sma200" in df.columns else np.full(len(df), np.nan)
    atr = df["atr14"].values

    for t in range(departure_window, len(df)):
        if not all(np.isfinite(v) for v in [ema20[t], sma50[t], atr[t]]):
            continue
        if atr[t] <= 0:
            continue

        if ema20[t] >= sma50[t]:
            continue
        if np.isfinite(sma200[t]) and sma50[t] >= sma200[t]:
            continue

        min_close = float(np.min(closes[t - departure_window : t + 1]))
        if min_close > ema20[t] - departure_atr * atr[t]:
            continue

        touched_ema20 = lows[t] <= ema20[t] <= highs[t]
        if not touched_ema20:
            continue

        dist = abs(closes[t] - ema20[t])
        if dist > dist_atr_max * atr[t]:
            continue

        if closes[t] > ema20[t]:
            continue

        candle_range = max(highs[t] - lows[t], 1e-9)
        in_lower_half = closes[t] <= lows[t] + 0.5 * candle_range
        if not in_lower_half:
            continue

        if not _recent_volume_not_expanding(vols, t, lookback=3):
            continue

        if sig_swings is not None:
            recent_low = any(
                (s.type == "LOW") and (t - departure_window <= s.index < t)
                for s in sig_swings
            )
            if not recent_low:
                continue

        stop = float(max(ema20[t] + atr[t], np.max(highs[max(0, t - 2) : t + 1]) + 0.2 * atr[t]))
        risk = stop - closes[t]
        if risk <= 0:
            continue

        out.append(EntrySignal(
            index=t,
            date=df["date"].iloc[t],
            price=round(closes[t], 2),
            type="TREND_SHORT_20EMA_RALLY",
            stop=round(stop, 2),
            target=round(closes[t] - risk * target_rr, 2),
            rr=target_rr,
            metadata={
                "setup_class": "TREND_SHORT_20EMA_RALLY",
                "direction": "SHORT",
                "key_level": round(ema20[t], 2),
                "ema20": round(ema20[t], 2),
                "sma50": round(sma50[t], 2),
                "dist_atr": round(dist / atr[t], 2),
                "departure_atr": round((ema20[t] - min_close) / atr[t], 2),
                "touched_ema20": touched_ema20,
                "in_lower_half": in_lower_half,
            },
        ))

    return out


def detect_base_ma_long(
    df: pd.DataFrame,
    base_regions: list[dict],
    dist_atr_max: float = 1.0,
    target_rr: float = 3.0,
) -> list[EntrySignal]:
    """
    Base Moving Average Long: EMA20 pullback inside a base region.
      - Bar must be within a base region
      - Close > SMA50
      - Wick touches EMA20 (low <= ema20 <= high)
      - Close within dist_atr_max * ATR of EMA20
    """
    out: list[EntrySignal] = []

    closes = df["close"].values
    highs = df["high"].values
    lows = df["low"].values
    ema20 = df["ema20"].values
    sma50 = df["sma50"].values
    atr = df["atr14"].values

    in_base = np.zeros(len(df), dtype=bool)
    for b in base_regions:
        s, e = int(b["start_idx"]), int(b["end_idx"])
        in_base[s : e + 1] = True

    for t in range(1, len(df)):
        if not in_base[t]:
            continue
        if not all(np.isfinite(v) for v in [ema20[t], sma50[t], atr[t]]):
            continue
        if atr[t] <= 0:
            continue

        if closes[t] <= sma50[t]:
            continue

        touched_ema20 = lows[t] <= ema20[t] <= highs[t]
        if not touched_ema20:
            continue

        dist = abs(closes[t] - ema20[t])
        if dist > dist_atr_max * atr[t]:
            continue

        stop = float(min(ema20[t] - atr[t], np.min(lows[max(0, t - 2) : t + 1]) - 0.2 * atr[t]))
        risk = closes[t] - stop
        if risk <= 0:
            continue

        out.append(EntrySignal(
            index=t,
            date=df["date"].iloc[t],
            price=round(closes[t], 2),
            type="BASE_MA_LONG",
            stop=round(stop, 2),
            target=round(closes[t] + risk * target_rr, 2),
            rr=target_rr,
            metadata={
                "setup_class": "BASE_MA_LONG",
                "direction": "LONG",
                "key_level": round(ema20[t], 2),
                "ema20": round(ema20[t], 2),
                "sma50": round(sma50[t], 2),
                "dist_atr": round(dist / atr[t], 2),
                "touched_ema20": touched_ema20,
            },
        ))

    return out


# ── Tunable parameters ──
SMA50_DIST_ATR   = 1.0   # max distance to SMA50 in ATR multiples
SMA50_VOL_BARS   = 3     # volume must decline over this many bars
SMA50_STOP_LOOK  = 5     # bars to look back for stop loss
SMA50_TARGET_RR  = 5.0   # risk-reward target

EMA20_DIST_ATR   = 1.0
EMA20_DEPART_ATR = 2.0   # minimum departure from EMA in ATR multiples
EMA20_DEPART_WIN = 30    # bars to look back for departure
EMA20_TARGET_RR  = 3.0

sma50_signals = detect_sma50_pullback(
    df,
    dist_atr_max=SMA50_DIST_ATR,
    vol_decline_bars=SMA50_VOL_BARS,
    stop_lookback=SMA50_STOP_LOOK,
    target_rr=SMA50_TARGET_RR,
)

ema20_signals = detect_ema20_pullback(
    df,
    dist_atr_max=EMA20_DIST_ATR,
    departure_atr=EMA20_DEPART_ATR,
    departure_window=EMA20_DEPART_WIN,
    target_rr=EMA20_TARGET_RR,
)

print(f"SMA50 pullback signals: {len(sma50_signals)}")
print(f"EMA20 pullback signals: {len(ema20_signals)}")

SMA50 pullback signals: 21
EMA20 pullback signals: 48


## 7 — Interactive Chart

In [9]:
# def build_chart(
#     df: pd.DataFrame,
#     fractal: list[SwingPoint],
#     sig: list[SwingPoint],
#     levels: list[Level],
#     sma50_sigs: list[EntrySignal],
#     ema20_sigs: list[EntrySignal],
#     title_suffix: str = "",
# ) -> go.Figure:
#     fig = make_subplots(
#         rows=2, cols=1, shared_xaxes=True,
#         row_heights=[0.75, 0.25], vertical_spacing=0.03,
#     )

#     # Candlestick
#     fig.add_trace(go.Candlestick(
#         x=df["date"], open=df["open"], high=df["high"],
#         low=df["low"], close=df["close"], name="Price",
#     ), row=1, col=1)

#     # Moving averages
#     for col, color, name in [
#         ("ema20", "#f59e0b", "EMA 20"),
#         ("sma50", "#3b82f6", "SMA 50"),
#         ("sma200", "#ef4444", "SMA 200"),
#     ]:
#         if col in df.columns:
#             fig.add_trace(go.Scatter(
#                 x=df["date"], y=df[col], name=name,
#                 line=dict(color=color, width=1.2),
#             ), row=1, col=1)

#     # Fractal pivots (already pre-filtered to active points)
#     fh = [p for p in fractal if p.type == "HIGH"]
#     fl = [p for p in fractal if p.type == "LOW"]
#     if fh:
#         fig.add_trace(go.Scatter(
#             x=[df["date"].iloc[p.index] for p in fh],
#             y=[p.price for p in fh],
#             mode="markers", name="Fractal High",
#             marker=dict(symbol="triangle-down", size=7, color="#f43f5e"),
#         ), row=1, col=1)
#     if fl:
#         fig.add_trace(go.Scatter(
#             x=[df["date"].iloc[p.index] for p in fl],
#             y=[p.price for p in fl],
#             mode="markers", name="Fractal Low",
#             marker=dict(symbol="triangle-up", size=7, color="#22c55e"),
#         ), row=1, col=1)

#     # Significant swings (already pre-filtered to active points)
#     sh = [p for p in sig if p.type == "HIGH"]
#     sl = [p for p in sig if p.type == "LOW"]
#     if sh:
#         fig.add_trace(go.Scatter(
#             x=[df["date"].iloc[p.index] for p in sh],
#             y=[p.price for p in sh],
#             mode="markers", name="Sig Swing High",
#             marker=dict(symbol="star", size=12, color="#dc2626", line=dict(width=1, color="white")),
#         ), row=1, col=1)
#     if sl:
#         fig.add_trace(go.Scatter(
#             x=[df["date"].iloc[p.index] for p in sl],
#             y=[p.price for p in sl],
#             mode="markers", name="Sig Swing Low",
#             marker=dict(symbol="star", size=12, color="#16a34a", line=dict(width=1, color="white")),
#         ), row=1, col=1)

#     # S/R levels as horizontal lines
#     for lv in levels:
#         color = "rgba(239,68,68,0.4)" if lv.type == "RESISTANCE" else "rgba(34,197,94,0.4)"
#         fig.add_hline(
#             y=lv.price, line_dash="dash", line_color=color, line_width=1,
#             annotation_text=f"{lv.type[0]} ${lv.price:.2f} (x{lv.touches})",
#             annotation_position="right", row=1, col=1,
#         )

#     # Entry signals (expected: current-bar valid only)
#     for sigs, name, color, symbol in [
#         (sma50_sigs, "SMA50 Setup (Current)", "#3b82f6", "diamond"),
#         (ema20_sigs, "EMA20 Setup (Current)", "#f59e0b", "diamond"),
#     ]:
#         if sigs:
#             fig.add_trace(go.Scatter(
#                 x=[df["date"].iloc[s.index] for s in sigs],
#                 y=[s.price for s in sigs],
#                 mode="markers", name=name,
#                 marker=dict(symbol=symbol, size=14, color=color,
#                             line=dict(width=2, color="white")),
#                 text=[f"Stop: {s.stop}  Target: {s.target}  RR: {s.rr}" for s in sigs],
#             ), row=1, col=1)

#     # Volume
#     colors = ["#22c55e" if df["close"].iloc[i] >= df["open"].iloc[i] else "#ef4444"
#               for i in range(len(df))]
#     fig.add_trace(go.Bar(
#         x=df["date"], y=df["volume"], name="Volume",
#         marker_color=colors, opacity=0.5,
#     ), row=2, col=1)

#     fig.update_layout(
#         title=f"{TICKER} — Setup Detectors{title_suffix}",
#         template="plotly_dark",
#         height=820,
#         xaxis_rangeslider_visible=False,
#         legend=dict(orientation="h", y=1.02, x=0),
#     )
#     fig.update_yaxes(title_text="Price", row=1, col=1)
#     fig.update_yaxes(title_text="Volume", row=2, col=1)
#     return fig


# chart = build_chart(df, fractal_pivots, sig_swings, sr_levels, [], [], title_suffix=" (All Data)")
# chart.show(renderer="notebook_connected")

C:\Users\perry\anaconda3\Lib\site-packages\kaleido\_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




## 8 — Signal Summary Table

In [11]:
all_signals = sma50_signals + ema20_signals
if all_signals:
    sig_df = pd.DataFrame([
        {"date": s.date, "type": s.type, "price": s.price,
         "stop": s.stop, "target": s.target, "rr": s.rr, **s.metadata}
        for s in all_signals
    ]).sort_values("date", ascending=False)
    display(sig_df.head(20))
else:
    print("No entry signals detected with current parameters.")

,date,type,price,stop,target,rr,dist_atr,sma50,setup_class,direction,key_level,ema20,departure_atr,touched_ema20
68,2026-01-15 00:00:00-05:00,EMA20_TREND_FOLLOW_BULL_PULLBACK,177.07,172.85,189.73,3.0,0.40,NaN,TREND_FOLLOWING_20EMA_BULL_PULLBACK,LONG,179.85,179.85,2.05,True
67,2026-01-14 00:00:00-05:00,EMA20_TREND_FOLLOW_BULL_PULLBACK,178.40,173.27,193.80,3.0,0.25,NaN,TREND_FOLLOWING_20EMA_BULL_PULLBACK,LONG,180.14,180.14,2.04,True
66,2026-01-13 00:00:00-05:00,EMA20_TREND_FOLLOW_BULL_PULLBACK,178.96,173.74,194.63,3.0,0.21,NaN,TREND_FOLLOWING_20EMA_BULL_PULLBACK,LONG,180.32,180.32,2.10,True
65,2026-01-12 00:00:00-05:00,EMA20_TREND_FOLLOW_BULL_PULLBACK,179.41,173.77,196.32,3.0,0.16,NaN,TREND_FOLLOWING_20EMA_BULL_PULLBACK,LONG,180.47,180.47,2.05,True
20,2026-01-06 00:00:00-05:00,SMA50_PULLBACK,179.71,159.95,278.53,5.0,0.16,180.89,NaN,NaN,NaN,NaN,NaN,NaN
19,2025-12-31 00:00:00-05:00,SMA50_PULLBACK,177.75,171.16,210.68,5.0,0.50,181.21,NaN,NaN,NaN,NaN,NaN,NaN
18,2025-12-09 00:00:00-05:00,SMA50_PULLBACK,181.84,161.17,285.19,5.0,0.34,179.37,NaN,NaN,NaN,NaN,NaN,NaN
17,2025-12-04 00:00:00-05:00,SMA50_PULLBACK,177.92,154.16,296.71,5.0,0.15,179.18,NaN,NaN,NaN,NaN,NaN,NaN
16,2025-11-17 00:00:00-05:00,SMA50_PULLBACK,171.25,156.11,246.97,5.0,0.85,180.78,NaN,NaN,NaN,NaN,NaN,NaN
64,2025-11-14 00:00:00-05:00,EMA20_TREND_FOLLOW_BULL_PULLBACK,174.01,173.13,176.66,3.0,0.92,NaN,TREND_FOLLOWING_20EMA_BULL_PULLBACK,LONG,184.22,184.22,2.07,False


## 9 — Quick Back-test: Forward Returns After Signals

In [12]:
def forward_returns(df: pd.DataFrame, signals: list[EntrySignal], horizons: list[int] = [5, 10, 20]) -> pd.DataFrame:
    """Compute forward returns after each signal to gauge effectiveness."""
    rows = []
    closes = df["close"].values
    for s in signals:
        row = {"date": s.date, "type": s.type, "entry": s.price}
        for h in horizons:
            if s.index + h < len(closes):
                ret = (closes[s.index + h] - s.price) / s.price * 100
                row[f"{h}d_ret_%"] = round(ret, 2)
            else:
                row[f"{h}d_ret_%"] = None

        # Hit target?
        hit_target = False
        hit_stop = False
        for j in range(s.index + 1, min(s.index + max(horizons) + 1, len(df))):
            if df["high"].iloc[j] >= s.target:
                hit_target = True
                break
            if df["low"].iloc[j] <= s.stop:
                hit_stop = True
                break

        row["outcome"] = "TARGET" if hit_target else "STOPPED" if hit_stop else "OPEN"
        rows.append(row)

    return pd.DataFrame(rows)


if all_signals:
    fwd = forward_returns(df, all_signals)
    print(f"\nSignal outcomes (within {20}d window):")
    print(fwd["outcome"].value_counts().to_string())
    print()
    display(fwd)
else:
    print("No signals to back-test.")


Signal outcomes (within 20d window):
outcome
STOPPED    32
TARGET     20
OPEN       17



,date,type,entry,5d_ret_%,10d_ret_%,20d_ret_%,outcome
0,2025-01-08 00:00:00-05:00,SMA50_PULLBACK,68.23,1.48,15.76,62.47,TARGET
1,2025-01-10 00:00:00-05:00,SMA50_PULLBACK,67.26,6.71,12.16,73.43,TARGET
2,2025-01-13 00:00:00-05:00,SMA50_PULLBACK,64.98,12.45,23.47,73.31,TARGET
3,2025-01-14 00:00:00-05:00,SMA50_PULLBACK,65.91,16.63,21.01,78.11,TARGET
4,2025-01-15 00:00:00-05:00,SMA50_PULLBACK,68.14,15.91,19.20,73.04,TARGET
...,...,...,...,...,...,...,...
64,2025-11-14 00:00:00-05:00,EMA20_TREND_FOLLOW_BULL_PULLBACK,174.01,-11.01,-3.75,5.31,STOPPED
65,2026-01-12 00:00:00-05:00,EMA20_TREND_FOLLOW_BULL_PULLBACK,179.41,-6.06,-7.64,-22.24,STOPPED
66,2026-01-13 00:00:00-05:00,EMA20_TREND_FOLLOW_BULL_PULLBACK,178.96,-7.62,-12.08,-24.18,STOPPED
67,2026-01-14 00:00:00-05:00,EMA20_TREND_FOLLOW_BULL_PULLBACK,178.40,-7.01,-14.88,-27.62,STOPPED


## 10 — Parameter Tuning Playground

Re-run any detector with different parameters and see the results instantly.

In [13]:
# New runner: date range + active-only swings + current MA setup only

# Date range filter (set None for full range)
START_DATE = None        # e.g. "2025-01-01"
END_DATE = None          # e.g. "2026-02-15"

# Fractal pivot sensitivity
FRACTAL_LOOKAHEAD = 10

# Significant swing thresholds
SIG_LEFT = 3
SIG_RIGHT = 3
SIG_PROM_ATR = 1.5
SIG_DEPART_ATR = 2.5
SIG_DEPART_LOOK = 10
SIG_MIN_SEP = 7

# S/R clustering
MERGE_PCT = 0.015
MIN_TOUCHES = 2

# MA setup thresholds
SMA50_DIST_ATR = 1.0
SMA50_VOL_BARS = 3
SMA50_STOP_LOOK = 5
SMA50_TARGET_RR = 5.0

EMA20_DIST_ATR = 1.0
EMA20_DEPART_ATR = 2.0
EMA20_DEPART_WIN = 30
EMA20_TARGET_RR = 3.0


def apply_date_range(df_src: pd.DataFrame, start_date=None, end_date=None) -> pd.DataFrame:
    out = df_src.copy()
    if start_date is not None:
        out = out[out["date"] >= pd.to_datetime(start_date)]
    if end_date is not None:
        out = out[out["date"] <= pd.to_datetime(end_date)]
    return out.reset_index(drop=True)


def add_indicators(df_in: pd.DataFrame) -> pd.DataFrame:
    out = df_in.copy()
    out["atr14"] = atr_series(out, 14)
    out["abs20"] = average_bar_size(out, 20)
    out["ema20"] = out["close"].ewm(span=20, adjust=False).mean()
    out["sma50"] = out["close"].rolling(50).mean()
    out["sma200"] = out["close"].rolling(200).mean()
    return out


def split_active_expired_swings(
    swings: list[SwingPoint],
    df_ctx: pd.DataFrame,
    latest_atr: float,
    atr_buffer: float = 1.0,
) -> tuple[list[SwingPoint], list[SwingPoint], list[SwingPoint]]:
    """
    Split swings into active + expired_high + expired_low.

    Path-based expiration (not just current-price snapshot):
      - HIGH expires if any future HIGH breaches swing_high + atr_buffer*ATR_ref
      - LOW  expires if any future LOW breaches swing_low  - atr_buffer*ATR_ref

    ATR_ref is swing ATR when available; fallback to atr14 at swing bar; fallback to latest ATR.
    """
    active: list[SwingPoint] = []
    expired_high: list[SwingPoint] = []
    expired_low: list[SwingPoint] = []

    highs = df_ctx["high"].values
    lows = df_ctx["low"].values
    atr_arr = df_ctx["atr14"].values
    n = len(df_ctx)

    for s in swings:
        if s.index >= n - 1:
            active.append(s)
            continue

        atr_ref = s.atr if np.isfinite(s.atr) and s.atr > 0 else atr_arr[s.index]
        if not np.isfinite(atr_ref) or atr_ref <= 0:
            atr_ref = latest_atr if np.isfinite(latest_atr) and latest_atr > 0 else 0.0

        thr = atr_buffer * atr_ref

        if s.type == "HIGH":
            breach_level = s.price + thr
            breached = np.any(highs[s.index + 1 :] > breach_level)
            if breached:
                expired_high.append(s)
            else:
                active.append(s)
        else:
            breach_level = s.price - thr
            breached = np.any(lows[s.index + 1 :] < breach_level)
            if breached:
                expired_low.append(s)
            else:
                active.append(s)

    return active, expired_high, expired_low


# Use master data if available, otherwise current df
if "df_all" not in globals():
    df_all = df.copy()

df_run = apply_date_range(df_all, START_DATE, END_DATE)
if len(df_run) < 60:
    raise ValueError("Date range too small. Use at least ~60 bars.")

df_run = add_indicators(df_run)
latest_close = float(df_run["close"].iloc[-1])
latest_atr = float(df_run["atr14"].iloc[-1]) if np.isfinite(df_run["atr14"].iloc[-1]) else 0.0

# Detect swings
fractal_pivots = detect_fractal_pivots(df_run, lookahead=FRACTAL_LOOKAHEAD)
sig_swings = detect_significant_swings(
    df_run,
    left=SIG_LEFT,
    right=SIG_RIGHT,
    prom_atr=SIG_PROM_ATR,
    depart_atr=SIG_DEPART_ATR,
    depart_lookahead=SIG_DEPART_LOOK,
    min_swing_sep=SIG_MIN_SEP,
)

# Split swings into active + expired (ATR invalidated)
fractal_active, fractal_expired_high, fractal_expired_low = split_active_expired_swings(
    fractal_pivots, df_run, latest_atr, atr_buffer=1.0
)
sig_active, sig_expired_high, sig_expired_low = split_active_expired_swings(
    sig_swings, df_run, latest_atr, atr_buffer=1.0
)
expired_high_all = fractal_expired_high + sig_expired_high
expired_low_all = fractal_expired_low + sig_expired_low

# S/R from active swings
sr_levels = cluster_levels(fractal_active + sig_active, merge_pct=MERGE_PCT, min_touches=MIN_TOUCHES)

# Detect MA setups, keep current valid, and label past ones as expired
sma50_all = detect_sma50_pullback(df_run, SMA50_DIST_ATR, SMA50_VOL_BARS, SMA50_STOP_LOOK, SMA50_TARGET_RR)
ema20_all = detect_ema20_pullback(df_run, EMA20_DIST_ATR, EMA20_DEPART_ATR, EMA20_DEPART_WIN, EMA20_TARGET_RR)

last_idx = len(df_run) - 1
sma50_current = [s for s in sma50_all if s.index == last_idx]
ema20_current = [s for s in ema20_all if s.index == last_idx]

# Expired MA entries = historical entries not active on current bar
sma50_expired = [s for s in sma50_all if s.index < last_idx]
ema20_expired = [s for s in ema20_all if s.index < last_idx]

range_text = f" [{df_run['date'].iloc[0]} -> {df_run['date'].iloc[-1]}]"
print(f"Bars used: {len(df_run)}{range_text}")
print(f"Latest close: {latest_close:.2f}")
print(f"Latest ATR14: {latest_atr:.2f}")
print(f"Active fractal swings: {len(fractal_active)}")
print(f"Active significant swings: {len(sig_active)}")
print(f"Expired swing highs (1 ATR): {len(expired_high_all)}")
print(f"Expired swing lows (1 ATR): {len(expired_low_all)}")
print(f"Active S/R levels: {len(sr_levels)}")
print(f"Current SMA50 setup valid: {len(sma50_current) > 0}")
print(f"Current EMA20 setup valid: {len(ema20_current) > 0}")
print(f"Expired SMA50 entries: {len(sma50_expired)}")
print(f"Expired EMA20 entries: {len(ema20_expired)}")

# Keep globals updated for follow-up cells
df = df_run
sma50_signals = sma50_current
ema20_signals = ema20_current


def build_ma_entry_chart(
    df_plot: pd.DataFrame,
    sma_sigs: list[EntrySignal],
    ema_sigs: list[EntrySignal],
    sma_expired: list[EntrySignal],
    ema_expired: list[EntrySignal],
) -> go.Figure:
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.75, 0.25], vertical_spacing=0.03)

    fig.add_trace(go.Candlestick(
        x=df_plot["date"], open=df_plot["open"], high=df_plot["high"],
        low=df_plot["low"], close=df_plot["close"], name="Price"
    ), row=1, col=1)

    fig.add_trace(go.Scatter(x=df_plot["date"], y=df_plot["ema20"], name="EMA 20", line=dict(color="#f59e0b", width=1.6)), row=1, col=1)
    fig.add_trace(go.Scatter(x=df_plot["date"], y=df_plot["sma50"], name="SMA 50", line=dict(color="#3b82f6", width=1.6)), row=1, col=1)

    # Expired MA setups (historical, not current)
    if sma_expired:
        fig.add_trace(go.Scatter(
            x=[df_plot["date"].iloc[s.index] for s in sma_expired],
            y=[s.price for s in sma_expired],
            mode="markers", name="SMA50 Expired",
            marker=dict(symbol="x", size=9, color="rgba(59,130,246,0.55)"),
        ), row=1, col=1)

    if ema_expired:
        fig.add_trace(go.Scatter(
            x=[df_plot["date"].iloc[s.index] for s in ema_expired],
            y=[s.price for s in ema_expired],
            mode="markers", name="EMA20 Expired",
            marker=dict(symbol="x", size=9, color="rgba(245,158,11,0.55)"),
        ), row=1, col=1)

    # Show current valid setups (if any)
    if sma_sigs:
        s = sma_sigs[0]
        x0 = df_plot["date"].iloc[s.index]
        fig.add_trace(go.Scatter(
            x=[x0], y=[s.price], mode="markers", name="SMA50 Current Setup",
            marker=dict(symbol="diamond", size=14, color="#3b82f6", line=dict(width=2, color="white")),
        ), row=1, col=1)
        fig.add_hline(y=s.stop, line_dash="dot", line_color="#3b82f6", annotation_text=f"SMA50 stop {s.stop:.2f}", row=1, col=1)
        fig.add_hline(y=s.target, line_dash="dot", line_color="#60a5fa", annotation_text=f"SMA50 target {s.target:.2f}", row=1, col=1)

    if ema_sigs:
        s = ema_sigs[0]
        x0 = df_plot["date"].iloc[s.index]
        fig.add_trace(go.Scatter(
            x=[x0], y=[s.price], mode="markers", name="EMA20 Current Setup",
            marker=dict(symbol="diamond", size=14, color="#f59e0b", line=dict(width=2, color="white")),
        ), row=1, col=1)
        fig.add_hline(y=s.stop, line_dash="dot", line_color="#f59e0b", annotation_text=f"EMA20 stop {s.stop:.2f}", row=1, col=1)
        fig.add_hline(y=s.target, line_dash="dot", line_color="#fbbf24", annotation_text=f"EMA20 target {s.target:.2f}", row=1, col=1)

    vol_colors = ["#22c55e" if df_plot["close"].iloc[i] >= df_plot["open"].iloc[i] else "#ef4444" for i in range(len(df_plot))]
    fig.add_trace(go.Bar(x=df_plot["date"], y=df_plot["volume"], name="Volume", marker_color=vol_colors, opacity=0.45), row=2, col=1)

    fig.update_layout(
        title=f"{TICKER} — MA Entry Chart{range_text}",
        template="plotly_dark",
        height=760,
        xaxis_rangeslider_visible=False,
        legend=dict(orientation="h", y=1.02, x=0),
    )
    fig.update_yaxes(title_text="Price", row=1, col=1)
    fig.update_yaxes(title_text="Volume", row=2, col=1)
    return fig


# Chart 1: Full setup context (active swings + expired swings + MA states)
# Old chart removed (use final classification chart cell)
# chart = build_chart(
#     df_run,
#     fractal_active,
#     sig_active,
#     sr_levels,
#     sma50_current,
#     ema20_current,
#     title_suffix=range_text,
# )

# Old chart blocks removed. Use the final MA Setup Classification chart cell only.
print("Old chart blocks removed. Run the final MA Setup Classification cell only.")

Bars used: 500 [2024-04-15 00:00:00-04:00 -> 2026-04-13 00:00:00-04:00]
Latest close: 132.37
Latest ATR14: 8.27
Active fractal swings: 18
Active significant swings: 19
Expired swing highs (1 ATR): 18
Expired swing lows (1 ATR): 21
Active S/R levels: 7
Current SMA50 setup valid: False
Current EMA20 setup valid: False
Expired SMA50 entries: 21
Expired EMA20 entries: 48
Old chart blocks removed. Run the final MA Setup Classification cell only.


In [14]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  TUNE THESE — re-run this cell + the chart cell to compare ║
# ╚══════════════════════════════════════════════════════════════╝

# Fractal pivot sensitivity
FRACTAL_LOOKAHEAD = 10

# Significant swing thresholds
SIG_LEFT        = 3
SIG_RIGHT       = 3
SIG_PROM_ATR    = 1.5
SIG_DEPART_ATR  = 2.5
SIG_DEPART_LOOK = 10
SIG_MIN_SEP     = 7

# S/R clustering
MERGE_PCT   = 0.015
MIN_TOUCHES = 2

# MA entry detectors
SMA50_DIST_ATR   = 1.0
SMA50_VOL_BARS   = 3
SMA50_STOP_LOOK  = 5
SMA50_TARGET_RR  = 5.0

EMA20_DIST_ATR   = 1.0
EMA20_DEPART_ATR = 2.0
EMA20_DEPART_WIN = 30
EMA20_TARGET_RR  = 3.0

# ── Re-detect ──
fractal_pivots = detect_fractal_pivots(df, lookahead=FRACTAL_LOOKAHEAD)
sig_swings = detect_significant_swings(
    df, left=SIG_LEFT, right=SIG_RIGHT,
    prom_atr=SIG_PROM_ATR, depart_atr=SIG_DEPART_ATR,
    depart_lookahead=SIG_DEPART_LOOK, min_swing_sep=SIG_MIN_SEP,
)
sr_levels = cluster_levels(fractal_pivots + sig_swings, merge_pct=MERGE_PCT, min_touches=MIN_TOUCHES)
sma50_signals = detect_sma50_pullback(df, SMA50_DIST_ATR, SMA50_VOL_BARS, SMA50_STOP_LOOK, SMA50_TARGET_RR)
ema20_signals = detect_ema20_pullback(df, EMA20_DIST_ATR, EMA20_DEPART_ATR, EMA20_DEPART_WIN, EMA20_TARGET_RR)

print(f"Fractal pivots: {len(fractal_pivots)}")
print(f"Significant swings: {len(sig_swings)}")
print(f"S/R levels: {len(sr_levels)}")
print(f"SMA50 entries: {len(sma50_signals)}")
print(f"EMA20 entries: {len(ema20_signals)}")

print("Deprecated tuning chart removed. Use the final MA Setup Classification chart cell.")

Fractal pivots: 29
Significant swings: 47
S/R levels: 18
SMA50 entries: 21
EMA20 entries: 48
Deprecated tuning chart removed. Use the final MA Setup Classification chart cell.


In [15]:
# FINAL RUNNER v2: Daily Base (spec) + merged bases + all expired overlays
# Use THIS as the last chart cell.

START_DATE = None
END_DATE = None


def _slice_df(df_src: pd.DataFrame, start_date=None, end_date=None) -> pd.DataFrame:
    out = df_src.copy()
    if start_date is not None:
        out = out[out["date"] >= pd.to_datetime(start_date)]
    if end_date is not None:
        out = out[out["date"] <= pd.to_datetime(end_date)]
    return out.reset_index(drop=True)


def _ensure_indicators(df_in: pd.DataFrame) -> pd.DataFrame:
    out = df_in.copy()
    if "atr14" not in out.columns:
        out["atr14"] = atr_series(out, 14)
    if "abs20" not in out.columns:
        out["abs20"] = average_bar_size(out, 20)
    if "ema20" not in out.columns:
        out["ema20"] = out["close"].ewm(span=20, adjust=False).mean()
    if "sma50" not in out.columns:
        out["sma50"] = out["close"].rolling(50).mean()
    if "sma200" not in out.columns:
        out["sma200"] = out["close"].rolling(200).mean()
    return out


def split_swings_for_chart(
    swings: list[SwingPoint],
    df_ctx: pd.DataFrame,
    atr_buffer: float = 1.0,
) -> tuple[list[SwingPoint], list[SwingPoint], list[SwingPoint]]:
    """Return (active, expired_high, expired_low) using path-based ATR invalidation."""
    active, expired_high, expired_low = [], [], []
    highs = df_ctx["high"].values
    lows = df_ctx["low"].values
    atr_arr = df_ctx["atr14"].values

    for s in swings:
        if s.index >= len(df_ctx) - 1:
            active.append(s)
            continue

        atr_ref = s.atr if np.isfinite(s.atr) and s.atr > 0 else atr_arr[s.index]
        if not np.isfinite(atr_ref) or atr_ref <= 0:
            atr_ref = atr_arr[-1] if np.isfinite(atr_arr[-1]) and atr_arr[-1] > 0 else 0.0

        thr = atr_buffer * atr_ref
        if s.type == "HIGH":
            if np.any(highs[s.index + 1 :] > s.price + thr):
                expired_high.append(s)
            else:
                active.append(s)
        else:
            if np.any(lows[s.index + 1 :] < s.price - thr):
                expired_low.append(s)
            else:
                active.append(s)

    return active, expired_high, expired_low


def detect_daily_base_candidates(
    df: pd.DataFrame,
    swings: list[SwingPoint],
    min_base_duration: int = 20,
    max_retrace: float = 0.50,
    contraction_tail: int = 5,
    min_window_bars: int = 5,
    min_window_gap: int = 3,
) -> list[dict]:
    """
    Stateful BASE rule from swing high:
      - Start at swing HIGH (peak)
      - Stay in BASE until first invalidation:
          1) retrace > max_retrace
          2) close < sma200
          3) high > (peak + 1*ATR)
      - Keep candidate only if duration >= min_base_duration

    This flags the full base span (not only the downswing leg).
    """
    highs = df["high"].values
    lows = df["low"].values
    closes = df["close"].values
    sma200 = df["sma200"].values
    atr = df["atr14"].values

    n = len(df)
    peak_swings = sorted([s for s in swings if s.type == "HIGH"], key=lambda s: s.index)
    out: list[dict] = []

    for pk in peak_swings:
        p = pk.index
        peak_price = float(pk.price)

        if p >= n - min_base_duration:
            continue

        atr_ref = atr[p] if np.isfinite(atr[p]) and atr[p] > 0 else np.nan
        if not np.isfinite(atr_ref) or atr_ref <= 0:
            # without ATR we cannot apply breakout = peak + 1*ATR reliably
            continue

        breakout_level = peak_price + atr_ref
        span_low = peak_price
        last_valid_idx = None
        invalid_reason = None

        for t in range(p, n):
            span_low = min(span_low, float(lows[t]))
            retrace = (peak_price - span_low) / peak_price if peak_price > 0 else 1.0

            broke_retrace = retrace > max_retrace
            below_200 = (not np.isfinite(sma200[t])) or (closes[t] < sma200[t])
            broke_out = highs[t] > breakout_level

            if broke_retrace:
                invalid_reason = "retrace_gt_50pct"
                break
            if below_200:
                invalid_reason = "close_below_sma200"
                break
            if broke_out:
                invalid_reason = "breakout_above_peak_plus_1atr"
                break

            last_valid_idx = t

        if last_valid_idx is None:
            continue

        duration = last_valid_idx - p + 1
        if duration < min_base_duration:
            continue

        base_high = float(np.max(highs[p : last_valid_idx + 1]))
        base_low = float(np.min(lows[p : last_valid_idx + 1]))
        retrace = (peak_price - base_low) / peak_price if peak_price > 0 else 1.0

        mid = max(p + 1, last_valid_idx - contraction_tail + 1)
        c_high = float(np.max(highs[mid : last_valid_idx + 1]))
        c_low = float(np.min(lows[mid : last_valid_idx + 1]))
        base_h = max(base_high - base_low, 1e-9)
        c_ratio = (c_high - c_low) / base_h

        out.append({
            "peak_idx": int(p),
            "peak_price": round(peak_price, 2),
            "start_idx": int(p),
            "mid_idx": int(mid),
            "end_idx": int(last_valid_idx),
            "base_high": round(base_high, 2),
            "base_low": round(base_low, 2),
            "contraction_high": round(c_high, 2),
            "contraction_low": round(c_low, 2),
            "contraction_ratio": round(c_ratio, 2),
            "retrace_pct": round(retrace, 3),
            "duration": int(duration),
            "breakout_level": round(float(breakout_level), 2),
            "invalid_reason": invalid_reason if invalid_reason is not None else "active_to_end_of_data",
        })

    return out


def merge_overlapping_bases(candidates: list[dict], df: pd.DataFrame, contraction_tail: int = 5) -> list[dict]:
    if not candidates:
        return []

    # one representative per peak (latest valid window)
    by_peak: dict[int, dict] = {}
    for c in candidates:
        pk = c["peak_idx"]
        cur = by_peak.get(pk)
        if cur is None or c["end_idx"] > cur["end_idx"]:
            by_peak[pk] = c

    reps = sorted(by_peak.values(), key=lambda x: x["start_idx"])
    merged: list[dict] = []

    highs = df["high"].values
    lows = df["low"].values

    for c in reps:
        if not merged:
            merged.append(c.copy())
            continue

        last = merged[-1]
        t0 = max(last["start_idx"], c["start_idx"])
        t1 = min(last["end_idx"], c["end_idx"])
        time_overlap = max(0, t1 - t0 + 1)
        min_len = max(1, min(last["end_idx"] - last["start_idx"] + 1, c["end_idx"] - c["start_idx"] + 1))
        t_overlap_ratio = time_overlap / min_len

        p0 = max(last["base_low"], c["base_low"])
        p1 = min(last["base_high"], c["base_high"])
        price_overlap = max(0.0, p1 - p0)
        min_h = max(1e-9, min(last["base_high"] - last["base_low"], c["base_high"] - c["base_low"]))
        p_overlap_ratio = price_overlap / min_h

        if t_overlap_ratio >= 0.60 and p_overlap_ratio >= 0.60:
            ns = min(last["start_idx"], c["start_idx"])
            ne = max(last["end_idx"], c["end_idx"])
            nm = max(ns + 1, ne - contraction_tail + 1)

            b_high = float(np.max(highs[ns:ne + 1]))
            b_low = float(np.min(lows[ns:ne + 1]))
            c_high = float(np.max(highs[nm:ne + 1]))
            c_low = float(np.min(lows[nm:ne + 1]))
            c_ratio = (c_high - c_low) / max(b_high - b_low, 1e-9)

            merged[-1] = {
                **last,
                "start_idx": int(ns),
                "mid_idx": int(nm),
                "end_idx": int(ne),
                "base_high": round(b_high, 2),
                "base_low": round(b_low, 2),
                "contraction_high": round(c_high, 2),
                "contraction_low": round(c_low, 2),
                "contraction_ratio": round(c_ratio, 2),
                "duration": int(ne - ns + 1),
                "retrace_pct": min(last.get("retrace_pct", 1.0), c.get("retrace_pct", 1.0)),
            }
        else:
            merged.append(c.copy())

    return merged


def detect_vcp_from_bases(
    df: pd.DataFrame,
    base_regions: list[dict],
    swings: list[SwingPoint],
    contraction_ratio: float = 0.75,
) -> list[dict]:
    """
    VCP from BASE using swing points with the requested anchor order:
      - Base must exist first.
      - L1 = lowest swing LOW after the base peak.
      - H2 = first swing HIGH registered after L1.
      - L2 = swing LOW after H2.
      - first_retrace  = peak_price - L1.price
      - second_retrace = H2.price - L2.price
      - Valid if second_retrace <= contraction_ratio * first_retrace
        (25% smaller => contraction_ratio = 0.75)
      - Extension rule: extend L2 to later swing LOWs as long as ratio stays valid,
        and keep the latest valid L2.
    """
    out: list[dict] = []

    for b in base_regions:
        p = int(b["peak_idx"])
        s = int(b["start_idx"])
        e = int(b["end_idx"])
        peak_price = float(b["peak_price"])

        if e - s < 4:
            continue

        seq = sorted([sw for sw in swings if s <= sw.index <= e and sw.index > p], key=lambda sw: sw.index)
        if len(seq) < 3:
            continue

        lows_after_peak = [sw for sw in seq if sw.type == "LOW"]
        if not lows_after_peak:
            continue

        # Requested anchor: lowest swing LOW after peak.
        l1 = min(lows_after_peak, key=lambda sw: (sw.price, sw.index))

        highs_after_l1 = [sw for sw in seq if sw.type == "HIGH" and sw.index > l1.index]
        if not highs_after_l1:
            continue

        # Requested anchor: swing HIGH registered after L1 (use first one).
        h2 = highs_after_l1[0]

        first_retrace = peak_price - float(l1.price)
        if first_retrace <= 0:
            continue

        later_lows = [sw for sw in seq if sw.type == "LOW" and sw.index > h2.index]
        if not later_lows:
            continue

        valid_l2 = None
        valid_ratio = None
        valid_second_retrace = None

        # Extension: keep latest later LOW that still satisfies contraction.
        for l2_candidate in later_lows:
            second_retrace_candidate = float(h2.price) - float(l2_candidate.price)
            if second_retrace_candidate <= 0:
                continue

            ratio_candidate = second_retrace_candidate / first_retrace
            if ratio_candidate <= contraction_ratio:
                valid_l2 = l2_candidate
                valid_ratio = ratio_candidate
                valid_second_retrace = second_retrace_candidate

        if valid_l2 is None:
            continue

        l2 = valid_l2
        second_retrace = valid_second_retrace
        ratio = valid_ratio

        out.append({
            "kind": "VCP",
            "peak_idx": int(p),
            "peak_price": round(peak_price, 2),
            "start_idx": int(s),
            "mid_idx": int(h2.index),
            "end_idx": int(l2.index),
            "base_high": float(b["base_high"]),
            "base_low": float(b["base_low"]),
            "contraction_high": round(float(h2.price), 2),
            "contraction_low": round(float(l2.price), 2),
            "l1_idx": int(l1.index),
            "l1_price": round(float(l1.price), 2),
            "h2_idx": int(h2.index),
            "h2_price": round(float(h2.price), 2),
            "l2_idx": int(l2.index),
            "l2_price": round(float(l2.price), 2),
            "first_retrace": round(float(first_retrace), 3),
            "second_retrace": round(float(second_retrace), 3),
            "w2_w1_ratio": round(float(ratio), 4),
            "duration": int(l2.index - s + 1),
            "parent_base_peak_idx": int(p),
        })

    return out


# Source + indicators
src = df_all.copy() if "df_all" in globals() else df.copy()
print(f"Using source ticker: {DATA_TICKER if 'DATA_TICKER' in globals() else TICKER}")
df_cls = _ensure_indicators(_slice_df(src, START_DATE, END_DATE))
if len(df_cls) < 80:
    raise ValueError("Need at least ~80 bars.")

# Swing structure
sig_swings = detect_significant_swings(df_cls, left=3, right=3, prom_atr=1.5, depart_atr=2.5, depart_lookahead=10, min_swing_sep=7)

# Daily base detection + merge
base_raw = detect_daily_base_candidates(df_cls, sig_swings, min_base_duration=20, max_retrace=0.50, contraction_tail=5)
base_merged = merge_overlapping_bases(base_raw, df_cls, contraction_tail=5)


def ensure_base_lows_are_significant_swings(
    df: pd.DataFrame,
    swings: list[SwingPoint],
    base_regions: list[dict],
) -> list[SwingPoint]:
    """If base low is not a significant swing LOW, add a synthetic one at that exact lowest bar."""
    lows = df["low"].values
    atr_vals = df["atr14"].values

    out = list(swings)
    existing_low_idx = {s.index for s in out if s.type == "LOW"}

    for b in base_regions:
        s = int(b["start_idx"])
        e = int(b["end_idx"])
        if e <= s:
            continue

        low_idx = s + int(np.argmin(lows[s : e + 1]))
        low_price = float(lows[low_idx])

        if low_idx in existing_low_idx:
            continue

        atr_ref = float(atr_vals[low_idx]) if np.isfinite(atr_vals[low_idx]) else 0.0
        out.append(SwingPoint(index=low_idx, price=low_price, type="LOW", atr=atr_ref, prominence=0.0))
        existing_low_idx.add(low_idx)

    out.sort(key=lambda sw: sw.index)
    return out


# Enforce: lowest point in each base is a significant swing low.
sig_swings = ensure_base_lows_are_significant_swings(df_cls, sig_swings, base_merged)

# Re-split swings for chart after augmentation
sig_active, sig_exp_high, sig_exp_low = split_swings_for_chart(sig_swings, df_cls)

# VCP from base (second contraction leg must be 25% smaller)
vcp_raw = detect_vcp_from_bases(df_cls, base_merged, sig_swings, contraction_ratio=0.75)
vcp_merged = merge_overlapping_bases(vcp_raw, df_cls, contraction_tail=5)

# Setup signals (safe for fresh kernels)
EMA20_DIST_ATR = globals().get("EMA20_DIST_ATR", 1.0)
EMA20_DEPART_WIN = globals().get("EMA20_DEPART_WIN", 30)
EMA20_TARGET_RR = globals().get("EMA20_TARGET_RR", 3.0)


def _call_detector(name: str, *args, **kwargs):
    fn = globals().get(name)
    if fn is None:
        print(f"[warn] Missing detector: {name} -> using empty list")
        return []
    return fn(*args, **kwargs)


trend_long_strict_all = _call_detector(
    "detect_trend_long_ema20_pullback_strict",
    df_cls,
    sig_swings,
    dist_atr_max=EMA20_DIST_ATR,
    departure_atr=1.3,
    departure_window=EMA20_DEPART_WIN,
    target_rr=EMA20_TARGET_RR,
)
trend_long_legacy_all = _call_detector(
    "detect_ema20_pullback",
    df_cls,
    dist_atr_max=EMA20_DIST_ATR,
    departure_atr=1.3,
    departure_window=EMA20_DEPART_WIN,
    target_rr=EMA20_TARGET_RR,
)
for s in trend_long_legacy_all:
    s.type = "TREND_LONG_20EMA_LEGACY"
    s.metadata["setup_label"] = "Trend Long (20EMA Legacy)"
    s.metadata["direction"] = "LONG"

base_fail_short_all = _call_detector("detect_base_failure_rally_short", df_cls)
trend_short_all = _call_detector(
    "detect_trend_short_ema20_rally",
    df_cls,
    sig_swings,
    dist_atr_max=EMA20_DIST_ATR,
    departure_atr=2.0,
    departure_window=EMA20_DEPART_WIN,
)

# Filter trend long signals that fall inside base regions
_in_base = np.zeros(len(df_cls), dtype=bool)
for _b in base_merged:
    _s, _e = int(_b["start_idx"]), int(_b["end_idx"])
    _in_base[_s : _e + 1] = True
trend_long_strict_all = [s for s in trend_long_strict_all if not _in_base[s.index]]
trend_long_legacy_all = [s for s in trend_long_legacy_all if not _in_base[s.index]]

# Base Moving Average Long (replaces trend long inside bases)
base_ma_long_all = detect_base_ma_long(df_cls, base_merged, dist_atr_max=EMA20_DIST_ATR, target_rr=EMA20_TARGET_RR)

def _infer_direction(sig: EntrySignal) -> str:
    md = sig.metadata if isinstance(sig.metadata, dict) else {}
    d = str(md.get("direction", "")).upper()
    if d in {"LONG", "SHORT"}:
        return d
    t = sig.type.upper()
    if "SHORT" in t or "FAILURE" in t:
        return "SHORT"
    return "LONG"


def _key_level(sig: EntrySignal) -> float:
    md = sig.metadata if isinstance(sig.metadata, dict) else {}
    for k in ["key_level", "pivot_level", "support_level", "resistance_level", "trigger_level", "entry_level"]:
        v = md.get(k)
        if v is not None and np.isfinite(float(v)):
            return float(v)
    return float(sig.price)


def _wick_intercepts(df_ctx: pd.DataFrame, t: int, level: float) -> bool:
    return float(df_ctx["low"].iloc[t]) <= level <= float(df_ctx["high"].iloc[t])


def split_alert_and_entry_signals(
    df_ctx: pd.DataFrame,
    setup_name: str,
    source_signals: list[EntrySignal],
) -> tuple[list[EntrySignal], list[EntrySignal]]:
    alerts: list[EntrySignal] = []
    entries: list[EntrySignal] = []

    for s in source_signals:
        t = int(s.index)
        if t < 0 or t >= len(df_ctx):
            continue

        direction = _infer_direction(s)
        key_level = _key_level(s)
        md = dict(s.metadata) if isinstance(s.metadata, dict) else {}
        md.update({"setup_class": setup_name, "signal_origin": s.type, "direction": direction, "key_level": key_level})

        alerts.append(
            EntrySignal(
                index=t,
                date=s.date,
                price=float(key_level),
                type=f"{setup_name}_ALERT",
                stop=s.stop,
                target=s.target,
                rr=s.rr,
                metadata={**md, "signal_kind": "ALERT"},
            )
        )

        if _wick_intercepts(df_ctx, t, key_level):
            entries.append(
                EntrySignal(
                    index=t,
                    date=s.date,
                    price=float(key_level),
                    type=f"{setup_name}_ENTRY",
                    stop=s.stop,
                    target=s.target,
                    rr=s.rr,
                    metadata={**md, "signal_kind": "ENTRY"},
                )
            )

    return alerts, entries


def detect_double_top_bottom_alert_entry(
    df_ctx: pd.DataFrame,
    base_regions: list[dict],
    approach_atr_mult: float = 0.50,
    min_bar_spacing: int = 7,
    departure_atr_mult: float = 1.5,
) -> tuple[list[EntrySignal], list[EntrySignal], list[EntrySignal], list[EntrySignal], dict]:
    """
    Gated double top/bottom alert-entry:
      - Alert = approach (close within band of key level).
      - Entry = wick intercepts key level.
      - First alert must be >= min_bar_spacing bars from base anchor.
      - Subsequent alerts require BOTH:
          * >= min_bar_spacing bars since previous alert
          * intermediate departure >= departure_atr_mult * ATR
            (furthest close away from level between prev alert and candidate)
      - Entries are only emitted on bars where an alert was also emitted (gating applies).
    Returns (dt_alerts, dt_entries, db_alerts, db_entries, stats).
    """
    atr = df_ctx["atr14"].values
    closes = df_ctx["close"].values

    dt_alerts: list[EntrySignal] = []
    dt_entries: list[EntrySignal] = []
    db_alerts: list[EntrySignal] = []
    db_entries: list[EntrySignal] = []

    stats = {
        "dt_approached": 0, "dt_blocked_spacing": 0, "dt_blocked_departure": 0, "dt_emitted": 0,
        "db_approached": 0, "db_blocked_spacing": 0, "db_blocked_departure": 0, "db_emitted": 0,
    }

    def _check_departure_dt(level: float, from_idx: int, to_idx: int) -> tuple[float, float]:
        """Return (max_departure_abs, departure_atr_mult) for DT (resistance)."""
        if to_idx - from_idx <= 1:
            return 0.0, 0.0
        seg = closes[from_idx + 1 : to_idx]
        if len(seg) == 0:
            return 0.0, 0.0
        k = from_idx + 1 + int(np.argmin(seg))
        dep = level - float(closes[k])
        a = float(atr[k]) if np.isfinite(atr[k]) and atr[k] > 0 else 1e-9
        return dep, dep / a

    def _check_departure_db(level: float, from_idx: int, to_idx: int) -> tuple[float, float]:
        """Return (max_departure_abs, departure_atr_mult) for DB (support)."""
        if to_idx - from_idx <= 1:
            return 0.0, 0.0
        seg = closes[from_idx + 1 : to_idx]
        if len(seg) == 0:
            return 0.0, 0.0
        k = from_idx + 1 + int(np.argmax(seg))
        dep = float(closes[k]) - level
        a = float(atr[k]) if np.isfinite(atr[k]) and atr[k] > 0 else 1e-9
        return dep, dep / a

    for b in base_regions:
        s = int(b["start_idx"])
        e = int(b["end_idx"])
        if e <= s:
            continue

        peak_level = float(b.get("peak_price", b.get("base_high")))
        low_level = float(b.get("base_low"))
        peak_idx = int(b.get("peak_idx", s))

        low_anchor_idx = s + int(np.argmin(closes[s : e + 1]))

        dt_last_alert_idx: int | None = None
        db_last_alert_idx: int | None = None

        for t in range(s, e + 1):
            a = atr[t] if np.isfinite(atr[t]) and atr[t] > 0 else 0.0
            band = approach_atr_mult * a
            c = float(closes[t])

            # --- Double Top ---
            is_dt_approach = (abs(c - peak_level) <= band) if band > 0 else (c == peak_level)
            if is_dt_approach:
                stats["dt_approached"] += 1

                anchor = peak_idx if dt_last_alert_idx is None else dt_last_alert_idx
                bars_since = t - anchor

                if bars_since < min_bar_spacing:
                    stats["dt_blocked_spacing"] += 1
                else:
                    dep_abs, dep_mult = (0.0, 0.0)
                    rearm_passed = True

                    if dt_last_alert_idx is not None:
                        dep_abs, dep_mult = _check_departure_dt(peak_level, dt_last_alert_idx, t)
                        rearm_passed = dep_mult >= departure_atr_mult

                    if not rearm_passed:
                        stats["dt_blocked_departure"] += 1
                    else:
                        stats["dt_emitted"] += 1
                        md = {
                            "setup_class": "DOUBLE_TOP", "signal_kind": "ALERT",
                            "key_level": peak_level, "direction": "SHORT",
                            "parent_base_peak_idx": peak_idx,
                            "bars_since_last_alert": bars_since,
                            "max_departure_abs": round(dep_abs, 3),
                            "departure_atr_mult": round(dep_mult, 3),
                            "rearm_passed": rearm_passed,
                        }
                        dt_alerts.append(EntrySignal(index=t, date=df_ctx["date"].iloc[t], price=peak_level, type="DOUBLE_TOP_ALERT", stop=peak_level, target=peak_level, rr=0.0, metadata=md))

                        if _wick_intercepts(df_ctx, t, peak_level):
                            md_e = {**md, "signal_kind": "ENTRY"}
                            dt_entries.append(EntrySignal(index=t, date=df_ctx["date"].iloc[t], price=peak_level, type="DOUBLE_TOP_ENTRY", stop=peak_level, target=peak_level, rr=0.0, metadata=md_e))

                        dt_last_alert_idx = t

            # --- Double Bottom ---
            is_db_approach = (abs(c - low_level) <= band) if band > 0 else (c == low_level)
            if is_db_approach:
                stats["db_approached"] += 1

                anchor = low_anchor_idx if db_last_alert_idx is None else db_last_alert_idx
                bars_since = t - anchor

                if bars_since < min_bar_spacing:
                    stats["db_blocked_spacing"] += 1
                else:
                    dep_abs, dep_mult = (0.0, 0.0)
                    rearm_passed = True

                    if db_last_alert_idx is not None:
                        dep_abs, dep_mult = _check_departure_db(low_level, db_last_alert_idx, t)
                        rearm_passed = dep_mult >= departure_atr_mult

                    if not rearm_passed:
                        stats["db_blocked_departure"] += 1
                    else:
                        stats["db_emitted"] += 1
                        md = {
                            "setup_class": "DOUBLE_BOTTOM", "signal_kind": "ALERT",
                            "key_level": low_level, "direction": "LONG",
                            "parent_base_peak_idx": peak_idx,
                            "bars_since_last_alert": bars_since,
                            "max_departure_abs": round(dep_abs, 3),
                            "departure_atr_mult": round(dep_mult, 3),
                            "rearm_passed": rearm_passed,
                        }
                        db_alerts.append(EntrySignal(index=t, date=df_ctx["date"].iloc[t], price=low_level, type="DOUBLE_BOTTOM_ALERT", stop=low_level, target=low_level, rr=0.0, metadata=md))

                        if _wick_intercepts(df_ctx, t, low_level):
                            md_e = {**md, "signal_kind": "ENTRY"}
                            db_entries.append(EntrySignal(index=t, date=df_ctx["date"].iloc[t], price=low_level, type="DOUBLE_BOTTOM_ENTRY", stop=low_level, target=low_level, rr=0.0, metadata=md_e))

                        db_last_alert_idx = t

    return dt_alerts, dt_entries, db_alerts, db_entries, stats


# Convert all setup outputs into ALERT / ENTRY streams (no dedup)
trend_long_strict_alert, trend_long_strict_entry = split_alert_and_entry_signals(df_cls, "TREND_LONG_20EMA_PULLBACK", trend_long_strict_all)
trend_long_legacy_alert, trend_long_legacy_entry = split_alert_and_entry_signals(df_cls, "TREND_LONG_20EMA_LEGACY", trend_long_legacy_all)
base_fail_alert, base_fail_entry = split_alert_and_entry_signals(df_cls, "BASE_FAILURE_SHORT", base_fail_short_all)
trend_short_alert, trend_short_entry = split_alert_and_entry_signals(df_cls, "TREND_SHORT_20EMA_RALLY", trend_short_all)
base_ma_long_alert, base_ma_long_entry = split_alert_and_entry_signals(df_cls, "BASE_MA_LONG", base_ma_long_all)

double_top_alert, double_top_entry, double_bottom_alert, double_bottom_entry, dt_db_stats = detect_double_top_bottom_alert_entry(df_cls, base_merged, approach_atr_mult=0.50, min_bar_spacing=7, departure_atr_mult=1.0)

all_alerts = trend_long_strict_alert + trend_long_legacy_alert + base_fail_alert + trend_short_alert + base_ma_long_alert + double_top_alert + double_bottom_alert
all_entries = trend_long_strict_entry + trend_long_legacy_entry + base_fail_entry + trend_short_entry + base_ma_long_entry + double_top_entry + double_bottom_entry

print(f"Base raw: {len(base_raw)} | Base merged: {len(base_merged)}")
print(f"VCP raw: {len(vcp_raw)} | VCP merged: {len(vcp_merged)}")
print(f"Expired significant swings: high={len(sig_exp_high)} low={len(sig_exp_low)}")
print("Signal counts (ALERT / ENTRY)")
print(f"  TREND_LONG_20EMA_PULLBACK: {len(trend_long_strict_alert)} / {len(trend_long_strict_entry)}")
print(f"  TREND_LONG_20EMA_LEGACY:   {len(trend_long_legacy_alert)} / {len(trend_long_legacy_entry)}")
print(f"  BASE_FAILURE_SHORT:        {len(base_fail_alert)} / {len(base_fail_entry)}")
print(f"  TREND_SHORT_20EMA_RALLY:   {len(trend_short_alert)} / {len(trend_short_entry)}")
print(f"  BASE_MA_LONG:              {len(base_ma_long_alert)} / {len(base_ma_long_entry)}")
print(f"  DOUBLE_TOP:                {len(double_top_alert)} / {len(double_top_entry)}")
print(f"  DOUBLE_BOTTOM:             {len(double_bottom_alert)} / {len(double_bottom_entry)}")
print("DT/DB Gating Stats:")
print(f"  DT approached={dt_db_stats['dt_approached']}  blocked_spacing={dt_db_stats['dt_blocked_spacing']}  blocked_departure={dt_db_stats['dt_blocked_departure']}  emitted={dt_db_stats['dt_emitted']}")
print(f"  DB approached={dt_db_stats['db_approached']}  blocked_spacing={dt_db_stats['db_blocked_spacing']}  blocked_departure={dt_db_stats['db_blocked_departure']}  emitted={dt_db_stats['db_emitted']}")
print(f"  TOTAL:                     {len(all_alerts)} / {len(all_entries)}")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.76, 0.24], vertical_spacing=0.03)
fig.add_trace(go.Candlestick(x=df_cls["date"], open=df_cls["open"], high=df_cls["high"], low=df_cls["low"], close=df_cls["close"], name="Price"), row=1, col=1)
fig.add_trace(go.Scatter(x=df_cls["date"], y=df_cls["ema20"], name="EMA 20", line=dict(color="#f59e0b", width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=df_cls["date"], y=df_cls["sma50"], name="SMA 50", line=dict(color="#3b82f6", width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=df_cls["date"], y=df_cls["sma200"], name="SMA 200", line=dict(color="#ef4444", width=1.5)), row=1, col=1)

# VCP is shown using rectangles only (no per-bar VCP markers)

# swings active
for arr, typ, name, symbol, color, size in [
    ([s for s in sig_active if s.type=="HIGH"], "H", "Sig Swing High", "star", "#dc2626", 10),
    ([s for s in sig_active if s.type=="LOW"], "L", "Sig Swing Low", "star", "#16a34a", 10),
]:
    if arr:
        fig.add_trace(go.Scatter(x=[df_cls["date"].iloc[s.index] for s in arr], y=[s.price for s in arr], mode="markers", name=name, marker=dict(symbol=symbol, size=size, color=color, line=dict(width=1, color="white") if symbol=="star" else None)), row=1, col=1)

# swings expired (significant only)
if sig_exp_high:
    fig.add_trace(go.Scatter(x=[df_cls["date"].iloc[s.index] for s in sig_exp_high], y=[s.price for s in sig_exp_high], mode="markers", name="Expired Swing High", marker=dict(symbol="x", size=8, color="rgba(239,68,68,0.60)")), row=1, col=1)
if sig_exp_low:
    fig.add_trace(go.Scatter(x=[df_cls["date"].iloc[s.index] for s in sig_exp_low], y=[s.price for s in sig_exp_low], mode="markers", name="Expired Swing Low", marker=dict(symbol="x", size=8, color="rgba(34,197,94,0.60)")), row=1, col=1)

# merged base rectangles
DRAW_BASE_RECTANGLES = True
BASE_RECT_MAX = 20
if DRAW_BASE_RECTANGLES:
    for b in base_merged[-BASE_RECT_MAX:]:
        fig.add_shape(type="rect", x0=df_cls["date"].iloc[b["start_idx"]], x1=df_cls["date"].iloc[b["end_idx"]], y0=b["base_low"], y1=b["base_high"], xref="x", yref="y", line=dict(color="rgba(34,197,94,0.45)", width=1), fillcolor="rgba(34,197,94,0.08)", layer="below")
        fig.add_shape(type="rect", x0=df_cls["date"].iloc[b["mid_idx"]], x1=df_cls["date"].iloc[b["end_idx"]], y0=b["contraction_low"], y1=b["contraction_high"], xref="x", yref="y", line=dict(color="rgba(16,185,129,0.75)", width=1), fillcolor="rgba(16,185,129,0.16)", layer="below")
    fig.add_trace(go.Scatter(x=[df_cls["date"].iloc[-1]], y=[df_cls["close"].iloc[-1]], mode="markers", name="Daily Base (merged) + Contraction", marker=dict(symbol="square", size=10, color="rgba(16,185,129,0.35)", line=dict(width=1, color="rgba(16,185,129,0.9)")), visible="legendonly"), row=1, col=1)

# merged VCP rectangles (blue)
DRAW_VCP_RECTANGLES = True
VCP_RECT_MAX = 20
if DRAW_VCP_RECTANGLES:
    for v in vcp_merged[-VCP_RECT_MAX:]:
        fig.add_shape(type="rect", x0=df_cls["date"].iloc[v["start_idx"]], x1=df_cls["date"].iloc[v["end_idx"]], y0=v["base_low"], y1=v["base_high"], xref="x", yref="y", line=dict(color="rgba(59,130,246,0.55)", width=1), fillcolor="rgba(59,130,246,0.08)", layer="below")
        fig.add_shape(type="rect", x0=df_cls["date"].iloc[v["h2_idx"]], x1=df_cls["date"].iloc[v["l2_idx"]], y0=v["l2_price"], y1=v["h2_price"], xref="x", yref="y", line=dict(color="rgba(37,99,235,0.9)", width=1), fillcolor="rgba(37,99,235,0.18)", layer="below")
    fig.add_trace(go.Scatter(x=[df_cls["date"].iloc[-1]], y=[df_cls["close"].iloc[-1]], mode="markers", name="VCP (W2 <= 0.75*W1)", marker=dict(symbol="square", size=10, color="rgba(37,99,235,0.35)", line=dict(width=1, color="rgba(37,99,235,0.95)")), visible="legendonly"), row=1, col=1)

ALERT_ENTRY_STYLE = {
    "TREND_LONG_20EMA_PULLBACK": {"alert": ("circle-open", "#10b981", "Trend Long Strict Alert"), "entry": ("diamond", "#10b981", "Trend Long Strict Entry")},
    "TREND_LONG_20EMA_LEGACY": {"alert": ("circle-open", "#84cc16", "Trend Long Legacy Alert"), "entry": ("diamond", "#84cc16", "Trend Long Legacy Entry")},
    "BASE_FAILURE_SHORT": {"alert": ("triangle-down-open", "#ef4444", "Base Failure Alert"), "entry": ("triangle-down", "#ef4444", "Base Failure Entry")},
    "TREND_SHORT_20EMA_RALLY": {"alert": ("circle-open", "#f97316", "Trend Short Alert"), "entry": ("diamond", "#f97316", "Trend Short Entry")},
    "BASE_MA_LONG": {"alert": ("star-open", "#38bdf8", "Base MA Long Alert"), "entry": ("star", "#38bdf8", "Base MA Long Entry")},
    "DOUBLE_TOP": {"alert": ("x", "#f43f5e", "Double Top Alert"), "entry": ("diamond-x", "#f43f5e", "Double Top Entry")},
    "DOUBLE_BOTTOM": {"alert": ("x", "#22c55e", "Double Bottom Alert"), "entry": ("diamond-x", "#22c55e", "Double Bottom Entry")},
}


def add_signal_group(sig_list: list[EntrySignal], setup_key: str, kind: str):
    if not sig_list:
        return
    symbol, color, name = ALERT_ENTRY_STYLE[setup_key][kind]
    size = 9 if kind == "alert" else 12
    width = 1 if kind == "alert" else 2
    fig.add_trace(
        go.Scatter(
            x=[df_cls["date"].iloc[s.index] for s in sig_list],
            y=[s.price for s in sig_list],
            mode="markers",
            name=name,
            marker=dict(symbol=symbol, size=size, color=color, line=dict(width=width, color="white")),
        ),
        row=1,
        col=1,
    )


add_signal_group(trend_long_strict_alert, "TREND_LONG_20EMA_PULLBACK", "alert")
add_signal_group(trend_long_strict_entry, "TREND_LONG_20EMA_PULLBACK", "entry")
add_signal_group(trend_long_legacy_alert, "TREND_LONG_20EMA_LEGACY", "alert")
add_signal_group(trend_long_legacy_entry, "TREND_LONG_20EMA_LEGACY", "entry")
add_signal_group(base_fail_alert, "BASE_FAILURE_SHORT", "alert")
add_signal_group(base_fail_entry, "BASE_FAILURE_SHORT", "entry")
add_signal_group(trend_short_alert, "TREND_SHORT_20EMA_RALLY", "alert")
add_signal_group(trend_short_entry, "TREND_SHORT_20EMA_RALLY", "entry")
add_signal_group(base_ma_long_alert, "BASE_MA_LONG", "alert")
add_signal_group(base_ma_long_entry, "BASE_MA_LONG", "entry")
add_signal_group(double_top_alert, "DOUBLE_TOP", "alert")
add_signal_group(double_top_entry, "DOUBLE_TOP", "entry")
add_signal_group(double_bottom_alert, "DOUBLE_BOTTOM", "alert")
add_signal_group(double_bottom_entry, "DOUBLE_BOTTOM", "entry")

vol_colors = ["#22c55e" if df_cls["close"].iloc[i] >= df_cls["open"].iloc[i] else "#ef4444" for i in range(len(df_cls))]
fig.add_trace(go.Bar(x=df_cls["date"], y=df_cls["volume"], name="Volume", marker_color=vol_colors, opacity=0.45), row=2, col=1)

fig.update_layout(title=f"{TICKER} — Daily Base + All Expired Overlays", template="plotly_dark", height=860, xaxis_rangeslider_visible=False, legend=dict(orientation="h", y=1.02, x=0))
fig.update_yaxes(title_text="Price", row=1, col=1)
fig.update_yaxes(title_text="Volume", row=2, col=1)
fig.show(renderer="notebook_connected")

Using source ticker: PLTR
Base raw: 6 | Base merged: 3
VCP raw: 3 | VCP merged: 3
Expired significant swings: high=14 low=15
Signal counts (ALERT / ENTRY)
  TREND_LONG_20EMA_PULLBACK: 24 / 8
  TREND_LONG_20EMA_LEGACY:   58 / 31
  BASE_FAILURE_SHORT:        41 / 13
  TREND_SHORT_20EMA_RALLY:   0 / 0
  BASE_MA_LONG:              24 / 24
  DOUBLE_TOP:                4 / 2
  DOUBLE_BOTTOM:             0 / 0
DT/DB Gating Stats:
  DT approached=8  blocked_spacing=4  blocked_departure=0  emitted=4
  DB approached=0  blocked_spacing=0  blocked_departure=0  emitted=0
  TOTAL:                     151 / 78


In [16]:
# ## Batch Setup Review — Export per-setup chart images for labeling

# Run this section to generate PNG chart images for each detected setup across
# multiple tickers and half-year windows (5 years). Images are saved to
# `artifacts/setup_review/{rule_version}/` with a `manifest.json` sidecar.

In [17]:
import json, os, time, traceback
from pathlib import Path
from datetime import datetime

REVIEW_TICKERS = [
  "ENPH", "SEDG", "RUN", "TWLO", "OKTA", "MDB", "ROKU", "PINS", "ETSY",
  "ZM", "DDOG", "CRWD", "NIO", "XPEV", "LI", "PLUG", "DVN", "MRO", "FANG",
  "CF", "MOS", "NTR", "ZIM", "DAC", "GSL", "AR", "HES", "MUR", "BTU",
  "ARCH", "CEIX", "AVAV", "KTOS", "HII", "SMCI", "ANET", "MU", "PLTR",
  "AI", "SNOW", "ZS", "FTNT", "VRT", "ETN", "HUBB", "VST", "CEG", "TLN",
  "MSTR", "COIN", "CLSK", "PWR", "NVT", "CCJ", "UEC", "LEU", "QQQ", "GLD",
  "SLV"
]

RULE_VERSION = "v1"
REVIEW_YEARS = 5
FETCH_PERIOD = "5y"
BARS_BEFORE_ALERT = 60
BARS_AFTER_ALERT = 20
MIN_WINDOW_BARS = 80
EXPORT_IMAGE_SCALE = 2

# Fast sanity mode for quick end-to-end validation
FAST_SANITY_MODE = False
SANITY_TICKER = "ENPH"
SANITY_YEARS = 1
SANITY_MAX_WINDOWS = 2
SANITY_MAX_EXPORTS_PER_WINDOW = 8

if FAST_SANITY_MODE:
    REVIEW_TICKERS = [SANITY_TICKER]
    REVIEW_YEARS = SANITY_YEARS
    EXPORT_IMAGE_SCALE = 1

ARTIFACTS_ROOT = Path("../artifacts/setup_review") / RULE_VERSION

rule_config_path = Path("rule_configs") / f"{RULE_VERSION}.json"
if rule_config_path.exists():
    with open(rule_config_path) as _rc:
        _cfg = json.load(_rc).get("parameters", {})
    for _k, _v in _cfg.items():
        if _k in globals():
            globals()[_k] = _v
    print(f"Loaded rule config from {rule_config_path}: {len(_cfg)} params")
else:
    print(f"No rule config at {rule_config_path}, using notebook defaults")

SETUP_BADGE_COLORS = {
    "TREND_LONG_20EMA_PULLBACK": "#10b981",
    "TREND_LONG_20EMA_LEGACY":   "#84cc16",
    "BASE_FAILURE_SHORT":        "#ef4444",
    "TREND_SHORT_20EMA_RALLY":   "#f97316",
    "BASE_MA_LONG":              "#38bdf8",
    "DOUBLE_TOP":                "#f43f5e",
    "DOUBLE_BOTTOM":             "#22c55e",
}


def generate_half_year_windows(df_src: pd.DataFrame, years: int = 5):
    end = df_src["date"].max()
    start = end - pd.DateOffset(years=years)
    windows = []
    cursor = pd.Timestamp(start)
    while cursor < end:
        w_end = cursor + pd.DateOffset(months=6)
        windows.append((cursor, min(w_end, pd.Timestamp(end))))
        cursor = w_end
    return windows


def build_per_setup_chart(
    df_ctx: pd.DataFrame,
    signal: EntrySignal,
    setup_type: str,
    ticker: str,
) -> go.Figure:
    idx = int(signal.index)
    i_start = max(0, idx - BARS_BEFORE_ALERT)
    i_end = min(len(df_ctx) - 1, idx + BARS_AFTER_ALERT)
    sl = df_ctx.iloc[i_start : i_end + 1].copy()

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.78, 0.22], vertical_spacing=0.03)

    fig.add_trace(go.Candlestick(
        x=sl["date"], open=sl["open"], high=sl["high"], low=sl["low"], close=sl["close"], name="Price",
    ), row=1, col=1)

    if "ema20" in sl.columns:
        fig.add_trace(go.Scatter(x=sl["date"], y=sl["ema20"], name="EMA 20", line=dict(color="#f59e0b", width=1.5)), row=1, col=1)
    if "sma50" in sl.columns:
        fig.add_trace(go.Scatter(x=sl["date"], y=sl["sma50"], name="SMA 50", line=dict(color="#3b82f6", width=1.5)), row=1, col=1)
    if "sma200" in sl.columns:
        fig.add_trace(go.Scatter(x=sl["date"], y=sl["sma200"], name="SMA 200", line=dict(color="#ef4444", width=1.2)), row=1, col=1)

    badge_color = SETUP_BADGE_COLORS.get(setup_type, "#a3a3a3")
    fig.add_trace(go.Scatter(
        x=[df_ctx["date"].iloc[idx]], y=[signal.price],
        mode="markers+text", name="Alert",
        marker=dict(symbol="circle-open", size=14, color=badge_color, line=dict(width=2.5)),
        text=[setup_type.replace("_", " ")],
        textposition="top center",
        textfont=dict(size=10, color=badge_color),
    ), row=1, col=1)

    vol_colors = ["#22c55e" if sl["close"].iloc[i] >= sl["open"].iloc[i] else "#ef4444" for i in range(len(sl))]
    fig.add_trace(go.Bar(x=sl["date"], y=sl["volume"], name="Volume", marker_color=vol_colors, opacity=0.45), row=2, col=1)

    alert_date_str = str(signal.date)[:10] if signal.date is not None else ""
    fig.update_layout(
        title=f"{ticker} — {setup_type.replace('_', ' ')} — {alert_date_str}",
        template="plotly_dark", height=540, width=1100,
        xaxis_rangeslider_visible=False,
        showlegend=False,
        margin=dict(l=50, r=30, t=50, b=30),
    )
    fig.update_yaxes(title_text="Price", row=1, col=1)
    fig.update_yaxes(title_text="Vol", row=2, col=1)
    return fig


print(f"Batch review config: {len(REVIEW_TICKERS)} tickers, {REVIEW_YEARS}y, rule={RULE_VERSION}")
print(f"Output dir: {ARTIFACTS_ROOT.resolve()}")

Loaded rule config from rule_configs\v1.json: 24 params
Batch review config: 59 tickers, 5y, rule=v1
Output dir: C:\Users\perry\Desktop\Alphaboard\artifacts\setup_review\v1


In [ ]:
def run_detection_pipeline(df_window: pd.DataFrame):
    """Run all detectors on a single window DataFrame. Returns list of (setup_type, signal)."""
    dfw = add_indicators(df_window)
    if len(dfw) < MIN_WINDOW_BARS:
        return [], dfw

    sig_sw = detect_significant_swings(
        dfw, left=SIG_LEFT, right=SIG_RIGHT,
        prom_atr=SIG_PROM_ATR, depart_atr=SIG_DEPART_ATR,
        depart_lookahead=SIG_DEPART_LOOK, min_swing_sep=SIG_MIN_SEP,
    )

    base_raw_w = detect_daily_base_candidates(dfw, sig_sw, min_base_duration=20, max_retrace=0.50, contraction_tail=5)
    base_merged_w = merge_overlapping_bases(base_raw_w, dfw, contraction_tail=5)
    sig_sw = ensure_base_lows_are_significant_swings(dfw, sig_sw, base_merged_w)

    _ema_dist = globals().get("EMA20_DIST_ATR", 1.0)
    _ema_dep_win = globals().get("EMA20_DEPART_WIN", 30)
    _ema_rr = globals().get("EMA20_TARGET_RR", 3.0)

    tl_strict = _call_detector("detect_trend_long_ema20_pullback_strict", dfw, sig_sw, dist_atr_max=_ema_dist, departure_atr=1.3, departure_window=_ema_dep_win, target_rr=_ema_rr)
    tl_legacy = _call_detector("detect_ema20_pullback", dfw, dist_atr_max=_ema_dist, departure_atr=1.3, departure_window=_ema_dep_win, target_rr=_ema_rr)
    for s in tl_legacy:
        s.type = "TREND_LONG_20EMA_LEGACY"
        s.metadata["setup_label"] = "Trend Long (20EMA Legacy)"
        s.metadata["direction"] = "LONG"

    bf_short = _call_detector("detect_base_failure_rally_short", dfw)
    ts_rally = _call_detector("detect_trend_short_ema20_rally", dfw, sig_sw, dist_atr_max=_ema_dist, departure_atr=2.0, departure_window=_ema_dep_win)

    _in_base_w = np.zeros(len(dfw), dtype=bool)
    for _b in base_merged_w:
        _s, _e = int(_b["start_idx"]), int(_b["end_idx"])
        _in_base_w[_s : _e + 1] = True
    tl_strict = [s for s in tl_strict if not _in_base_w[s.index]]
    tl_legacy = [s for s in tl_legacy if not _in_base_w[s.index]]

    bml = _call_detector("detect_base_ma_long", dfw, base_merged_w, dist_atr_max=_ema_dist, target_rr=_ema_rr)

    tl_strict_a, tl_strict_e = split_alert_and_entry_signals(dfw, "TREND_LONG_20EMA_PULLBACK", tl_strict)
    tl_legacy_a, tl_legacy_e = split_alert_and_entry_signals(dfw, "TREND_LONG_20EMA_LEGACY", tl_legacy)
    bf_a, bf_e = split_alert_and_entry_signals(dfw, "BASE_FAILURE_SHORT", bf_short)
    ts_a, ts_e = split_alert_and_entry_signals(dfw, "TREND_SHORT_20EMA_RALLY", ts_rally)
    bml_a, bml_e = split_alert_and_entry_signals(dfw, "BASE_MA_LONG", bml)

    dt_a, dt_e, db_a, db_e, _ = detect_double_top_bottom_alert_entry(
        dfw, base_merged_w, approach_atr_mult=0.50, min_bar_spacing=7, departure_atr_mult=1.0,
    )

    results = []
    for setup_type, alerts in [
        ("TREND_LONG_20EMA_PULLBACK", tl_strict_a),
        ("TREND_LONG_20EMA_LEGACY", tl_legacy_a),
        ("BASE_FAILURE_SHORT", bf_a),
        ("TREND_SHORT_20EMA_RALLY", ts_a),
        ("BASE_MA_LONG", bml_a),
        ("DOUBLE_TOP", dt_a),
        ("DOUBLE_BOTTOM", db_a),
    ]:
        for sig in alerts:
            results.append((setup_type, sig))

    return results, dfw


manifest = []
skipped_log = []
total_exported = 0

for ticker_i, ticker in enumerate(REVIEW_TICKERS):
    ticker_start_ts = time.time()
    print(f"\n[{ticker_i+1}/{len(REVIEW_TICKERS)}] Fetching {ticker}...", flush=True)
    try:
        raw_t = yf.Ticker(ticker).history(period=FETCH_PERIOD, interval="1d")
        if raw_t.empty:
            skipped_log.append(f"{ticker}: no data returned")
            print(f"  SKIP: no data")
            continue
        if isinstance(raw_t.columns, pd.MultiIndex):
            raw_t.columns = raw_t.columns.get_level_values(0)
        raw_t = raw_t[["Open", "High", "Low", "Close", "Volume"]].dropna().reset_index()
        raw_t.rename(columns={"Date": "date", "Datetime": "date"}, inplace=True)
        df_t = raw_t.rename(columns={"Open": "open", "High": "high", "Low": "low", "Close": "close", "Volume": "volume"})
    except Exception as e:
        skipped_log.append(f"{ticker}: fetch error - {e}")
        print(f"  SKIP: {e}")
        continue

    windows = generate_half_year_windows(df_t, years=REVIEW_YEARS)
    if FAST_SANITY_MODE:
        windows = windows[:SANITY_MAX_WINDOWS]
    print(f"  {len(windows)} windows, {len(df_t)} total bars", flush=True)

    for wi, (w_start, w_end) in enumerate(windows):
        print(f"    -> window {wi+1}/{len(windows)} [{str(w_start)[:10]} to {str(w_end)[:10]}]", flush=True)
        df_win = df_t[(df_t["date"] >= w_start) & (df_t["date"] <= w_end)].reset_index(drop=True)
        if len(df_win) < MIN_WINDOW_BARS:
            skipped_log.append(f"{ticker} window {wi}: only {len(df_win)} bars")
            continue

        try:
            detections, df_enriched = run_detection_pipeline(df_win)
            print(f"      detections: {len(detections)}", flush=True)
        except Exception as e:
            skipped_log.append(f"{ticker} window {wi}: pipeline error - {e}")
            traceback.print_exc()
            continue

        if FAST_SANITY_MODE and len(detections) > SANITY_MAX_EXPORTS_PER_WINDOW:
            detections = detections[:SANITY_MAX_EXPORTS_PER_WINDOW]
            print(f"      sanity cap applied: {len(detections)} exports", flush=True)

        w_start_str = str(w_start.date()) if hasattr(w_start, "date") else str(w_start)[:10]
        w_end_str = str(w_end.date()) if hasattr(w_end, "date") else str(w_end)[:10]

        for si, (setup_type, sig) in enumerate(detections):
            if si % 10 == 0:
                print(f"      exporting {si+1}/{len(detections)}", flush=True)
            alert_date_raw = sig.date
            if hasattr(alert_date_raw, "strftime"):
                alert_date_str = alert_date_raw.strftime("%Y-%m-%d")
            else:
                alert_date_str = str(alert_date_raw)[:10]

            chart_id = f"{ticker}_{setup_type}_{alert_date_str}"
            out_dir = ARTIFACTS_ROOT / ticker / setup_type
            out_dir.mkdir(parents=True, exist_ok=True)
            img_path = out_dir / f"{alert_date_str}.png"
            rel_path = str(img_path.relative_to(ARTIFACTS_ROOT))

            fig = build_per_setup_chart(df_enriched, sig, setup_type, ticker)
            try:
                from kaleido.scopes.plotly import PlotlyScope
                scope = PlotlyScope()
                img_bytes = scope.transform(fig, format="png", width=1100, height=540, scale=EXPORT_IMAGE_SCALE)
                with open(str(img_path), "wb") as f_img:
                    f_img.write(img_bytes)
            except Exception as e:
                try:
                    html_path = img_path.with_suffix(".html")
                    fig.write_html(str(html_path), include_plotlyjs="cdn")
                    rel_path = str(html_path.relative_to(ARTIFACTS_ROOT))
                    print(f"      png failed, saved html: {chart_id}", flush=True)
                except Exception as e2:
                    skipped_log.append(f"{ticker} window {wi}: export failed for {chart_id} - {e} / {e2}")
                    print(f"      export failed for {chart_id}: {e}", flush=True)
                    continue

            entry_sig = None
            md = sig.metadata if isinstance(sig.metadata, dict) else {}
            manifest.append({
                "chart_id": chart_id,
                "ticker": ticker,
                "setup_type": setup_type,
                "alert_date": alert_date_str,
                "alert_price": round(float(sig.price), 2),
                "window_start": w_start_str,
                "window_end": w_end_str,
                "rule_version": RULE_VERSION,
                "chart_path": rel_path,
                "direction": md.get("direction", "LONG"),
            })
            total_exported += 1

    ticker_exports = sum(1 for m in manifest if m['ticker'] == ticker)
    print(f"  Exported {ticker_exports} charts for {ticker} in {time.time() - ticker_start_ts:.1f}s", flush=True)

manifest_path = ARTIFACTS_ROOT / "manifest.json"
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

if skipped_log:
    with open(ARTIFACTS_ROOT / "skipped.log", "w") as f:
        f.write("\n".join(skipped_log))

print(f"\nDone! Exported {total_exported} charts across {len(REVIEW_TICKERS)} tickers.")
print(f"Manifest: {manifest_path.resolve()}")
print(f"Skipped: {len(skipped_log)} items")


[1/59] Fetching ENPH...
  10 windows, 1255 total bars
    -> window 1/10 [2021-04-13 to 2021-10-13]
      detections: 49
      exporting 1/49
      png failed, saved html: ENPH_TREND_LONG_20EMA_PULLBACK_2021-07-20
      png failed, saved html: ENPH_TREND_LONG_20EMA_PULLBACK_2021-07-21
      png failed, saved html: ENPH_TREND_LONG_20EMA_PULLBACK_2021-07-22
      png failed, saved html: ENPH_TREND_LONG_20EMA_PULLBACK_2021-07-23
      png failed, saved html: ENPH_TREND_LONG_20EMA_PULLBACK_2021-07-26
      png failed, saved html: ENPH_TREND_LONG_20EMA_PULLBACK_2021-08-09
      png failed, saved html: ENPH_TREND_LONG_20EMA_PULLBACK_2021-08-10
      png failed, saved html: ENPH_TREND_LONG_20EMA_PULLBACK_2021-08-11
      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2021-07-14
      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2021-07-15
      exporting 11/49
      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2021-07-19
      png failed, saved html: ENPH_TREND_LONG_20E

      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2022-09-06
      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2022-09-09
      exporting 31/39
      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2022-09-13
      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2022-09-27
      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2022-09-28
      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2022-09-29
      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2022-09-30
      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2022-10-03
      png failed, saved html: ENPH_BASE_FAILURE_SHORT_2022-10-04
      png failed, saved html: ENPH_BASE_FAILURE_SHORT_2022-10-06
      png failed, saved html: ENPH_BASE_FAILURE_SHORT_2022-10-12
    -> window 4/10 [2022-10-13 to 2023-04-13]
      detections: 51
      exporting 1/51
      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2022-12-22
      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2022-1

      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2024-01-11
      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2024-01-23
      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2024-01-24
      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2024-01-25
      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2024-02-20
      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2024-02-21
      exporting 21/46
      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2024-02-22
      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2024-02-23
      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2024-02-26
      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2024-02-27
      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2024-02-28
      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2024-02-29
      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2024-03-04
      png failed, saved html: ENPH_TREND_LONG_20EMA_LEGACY_2024-03-0

      png failed, saved html: ENPH_BASE_FAILURE_SHORT_2025-02-27
      png failed, saved html: ENPH_BASE_FAILURE_SHORT_2025-02-28
      png failed, saved html: ENPH_BASE_FAILURE_SHORT_2025-03-06
      png failed, saved html: ENPH_BASE_FAILURE_SHORT_2025-03-07
      png failed, saved html: ENPH_BASE_FAILURE_SHORT_2025-03-10
      png failed, saved html: ENPH_BASE_FAILURE_SHORT_2025-03-11
      png failed, saved html: ENPH_BASE_FAILURE_SHORT_2025-03-12
      png failed, saved html: ENPH_BASE_FAILURE_SHORT_2025-03-13
      exporting 41/56
      png failed, saved html: ENPH_BASE_FAILURE_SHORT_2025-03-14
      png failed, saved html: ENPH_BASE_FAILURE_SHORT_2025-03-18
      png failed, saved html: ENPH_BASE_FAILURE_SHORT_2025-03-19
      png failed, saved html: ENPH_BASE_FAILURE_SHORT_2025-03-20
      png failed, saved html: ENPH_BASE_FAILURE_SHORT_2025-03-21
      png failed, saved html: ENPH_BASE_FAILURE_SHORT_2025-03-24
      png failed, saved html: ENPH_BASE_FAILURE_SHORT_2025-03-26
   

      png failed, saved html: ENPH_BASE_FAILURE_SHORT_2026-04-02
  Exported 460 charts for ENPH in 214.5s

[2/59] Fetching SEDG...
  10 windows, 1255 total bars
    -> window 1/10 [2021-04-13 to 2021-10-13]
      detections: 64
      exporting 1/64
      png failed, saved html: SEDG_TREND_LONG_20EMA_PULLBACK_2021-07-06
      png failed, saved html: SEDG_TREND_LONG_20EMA_PULLBACK_2021-07-07
      png failed, saved html: SEDG_TREND_LONG_20EMA_PULLBACK_2021-07-09
      png failed, saved html: SEDG_TREND_LONG_20EMA_PULLBACK_2021-07-12
      png failed, saved html: SEDG_TREND_LONG_20EMA_PULLBACK_2021-07-13
      png failed, saved html: SEDG_TREND_LONG_20EMA_PULLBACK_2021-07-21
      png failed, saved html: SEDG_TREND_LONG_20EMA_PULLBACK_2021-07-22
      png failed, saved html: SEDG_TREND_LONG_20EMA_PULLBACK_2021-08-13
      png failed, saved html: SEDG_TREND_LONG_20EMA_PULLBACK_2021-08-16
      png failed, saved html: SEDG_TREND_LONG_20EMA_PULLBACK_2021-08-23
      exporting 11/64
      png

...

      png failed, saved html: SEDG_BASE_FAILURE_SHORT_2023-10-10
      png failed, saved html: SEDG_BASE_FAILURE_SHORT_2023-10-11
      png failed, saved html: SEDG_BASE_FAILURE_SHORT_2023-10-12
      png failed, saved html: SEDG_BASE_FAILURE_SHORT_2023-10-13
      exporting 21/21
      png failed, saved html: SEDG_TREND_SHORT_20EMA_RALLY_2023-07-05
    -> window 6/10 [2023-10-13 to 2024-04-13]
      detections: 45
      exporting 1/45
      png failed, saved html: SEDG_TREND_LONG_20EMA_PULLBACK_2023-12-28
      png failed, saved html: SEDG_TREND_LONG_20EMA_PULLBACK_2023-12-29
      png failed, saved html: SEDG_TREND_LONG_20EMA_PULLBACK_2024-01-02
      png failed, saved html: SEDG_TREND_LONG_20EMA_LEGACY_2023-12-28
      png failed, saved html: SEDG_TREND_LONG_20EMA_LEGACY_2023-12-29
      png failed, saved html: SEDG_TREND_LONG_20EMA_LEGACY_2024-01-02
      png failed, saved html: SEDG_TREND_LONG_20EMA_LEGACY_2024-01-03
      png failed, saved html: SEDG_BASE_FAILURE_SHORT_2024-01-22

      png failed, saved html: SEDG_TREND_LONG_20EMA_LEGACY_2025-01-28
      png failed, saved html: SEDG_TREND_LONG_20EMA_LEGACY_2025-01-29
      png failed, saved html: SEDG_TREND_LONG_20EMA_LEGACY_2025-01-30
      png failed, saved html: SEDG_TREND_LONG_20EMA_LEGACY_2025-01-31
      png failed, saved html: SEDG_TREND_LONG_20EMA_LEGACY_2025-02-27
      png failed, saved html: SEDG_TREND_LONG_20EMA_LEGACY_2025-02-28
      png failed, saved html: SEDG_TREND_LONG_20EMA_LEGACY_2025-03-04
      png failed, saved html: SEDG_TREND_LONG_20EMA_LEGACY_2025-03-05
      exporting 31/59
      png failed, saved html: SEDG_TREND_LONG_20EMA_LEGACY_2025-03-06
      png failed, saved html: SEDG_TREND_LONG_20EMA_LEGACY_2025-03-07
      png failed, saved html: SEDG_TREND_LONG_20EMA_LEGACY_2025-03-10
      png failed, saved html: SEDG_TREND_LONG_20EMA_LEGACY_2025-03-11
      png failed, saved html: SEDG_TREND_LONG_20EMA_LEGACY_2025-03-12
      png failed, saved html: SEDG_TREND_LONG_20EMA_LEGACY_2025-03-1

      png failed, saved html: SEDG_BASE_FAILURE_SHORT_2025-12-29
      png failed, saved html: SEDG_BASE_FAILURE_SHORT_2025-12-30
      png failed, saved html: SEDG_BASE_FAILURE_SHORT_2025-12-31
      exporting 31/35
      png failed, saved html: SEDG_BASE_FAILURE_SHORT_2026-01-06
      png failed, saved html: SEDG_BASE_FAILURE_SHORT_2026-01-07
      png failed, saved html: SEDG_BASE_FAILURE_SHORT_2026-01-08
      png failed, saved html: SEDG_BASE_FAILURE_SHORT_2026-01-09
      png failed, saved html: SEDG_BASE_FAILURE_SHORT_2026-01-20
  Exported 459 charts for SEDG in 191.6s

[3/59] Fetching RUN...
  10 windows, 1255 total bars
    -> window 1/10 [2021-04-13 to 2021-10-13]
      detections: 65
      exporting 1/65
      png failed, saved html: RUN_TREND_LONG_20EMA_PULLBACK_2021-07-07
      png failed, saved html: RUN_TREND_LONG_20EMA_PULLBACK_2021-07-12
      png failed, saved html: RUN_TREND_LONG_20EMA_PULLBACK_2021-07-13
      png failed, saved html: RUN_TREND_LONG_20EMA_PULLBACK_20

      png failed, saved html: RUN_TREND_LONG_20EMA_PULLBACK_2022-07-07
      png failed, saved html: RUN_TREND_LONG_20EMA_PULLBACK_2022-07-11
      png failed, saved html: RUN_TREND_LONG_20EMA_PULLBACK_2022-07-12
      png failed, saved html: RUN_TREND_LONG_20EMA_PULLBACK_2022-07-13
      png failed, saved html: RUN_TREND_LONG_20EMA_PULLBACK_2022-07-14
      png failed, saved html: RUN_TREND_LONG_20EMA_PULLBACK_2022-08-17
      exporting 11/59
      png failed, saved html: RUN_TREND_LONG_20EMA_PULLBACK_2022-08-19
      png failed, saved html: RUN_TREND_LONG_20EMA_PULLBACK_2022-08-25
      png failed, saved html: RUN_TREND_LONG_20EMA_PULLBACK_2022-08-26
      png failed, saved html: RUN_TREND_LONG_20EMA_PULLBACK_2022-08-29
      png failed, saved html: RUN_TREND_LONG_20EMA_PULLBACK_2022-08-31
      png failed, saved html: RUN_TREND_LONG_20EMA_PULLBACK_2022-09-06
      png failed, saved html: RUN_TREND_LONG_20EMA_PULLBACK_2022-09-15
      png failed, saved html: RUN_TREND_LONG_20EMA_PULL

      png failed, saved html: RUN_TREND_LONG_20EMA_LEGACY_2023-07-31
      png failed, saved html: RUN_TREND_LONG_20EMA_LEGACY_2023-08-01
      png failed, saved html: RUN_TREND_LONG_20EMA_LEGACY_2023-08-03
      png failed, saved html: RUN_BASE_FAILURE_SHORT_2023-06-26
      png failed, saved html: RUN_BASE_FAILURE_SHORT_2023-06-27
      png failed, saved html: RUN_BASE_FAILURE_SHORT_2023-06-30
      exporting 21/43
      png failed, saved html: RUN_BASE_FAILURE_SHORT_2023-07-07
      png failed, saved html: RUN_BASE_FAILURE_SHORT_2023-07-10
      png failed, saved html: RUN_BASE_FAILURE_SHORT_2023-08-09
      png failed, saved html: RUN_BASE_FAILURE_SHORT_2023-08-10
      png failed, saved html: RUN_BASE_FAILURE_SHORT_2023-08-11
      png failed, saved html: RUN_BASE_FAILURE_SHORT_2023-08-14
      png failed, saved html: RUN_BASE_FAILURE_SHORT_2023-08-29
      png failed, saved html: RUN_BASE_FAILURE_SHORT_2023-08-30
      png failed, saved html: RUN_BASE_FAILURE_SHORT_2023-09-05
   

In [ ]:
# --- Diagnostic: BASE + DOUBLE_TOP/DOUBLE_BOTTOM only ---
# Run this cell to verify whether base detection and DT/DB gating are working.

DIAG_TICKER = globals().get("TICKER", "PLTR")
DIAG_PERIOD = "5y"
DIAG_START = None   # e.g. "2023-01-01"
DIAG_END = None     # e.g. "2023-12-31"

# DT/DB gating knobs for diagnosis (same defaults as main pipeline)
DIAG_APPROACH_ATR_MULT = 0.50
DIAG_MIN_BAR_SPACING = 7
DIAG_DEPARTURE_ATR_MULT = 1.0

required_fns = [
    "add_indicators",
    "detect_significant_swings",
    "detect_daily_base_candidates",
    "merge_overlapping_bases",
    "ensure_base_lows_are_significant_swings",
    "detect_double_top_bottom_alert_entry",
]
missing = [fn for fn in required_fns if fn not in globals()]
if missing:
    raise RuntimeError(f"Missing required functions in kernel: {missing}. Run detector definition cells first.")

print(f"[diag] ticker={DIAG_TICKER} period={DIAG_PERIOD} start={DIAG_START} end={DIAG_END}")

raw_d = yf.Ticker(DIAG_TICKER).history(period=DIAG_PERIOD, interval="1d")
if raw_d.empty:
    raise ValueError(f"No data for {DIAG_TICKER}")
if isinstance(raw_d.columns, pd.MultiIndex):
    raw_d.columns = raw_d.columns.get_level_values(0)
raw_d = raw_d[["Open", "High", "Low", "Close", "Volume"]].dropna().reset_index()
raw_d.rename(columns={"Date": "date", "Datetime": "date"}, inplace=True)
df_diag = raw_d.rename(columns={
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Close": "close",
    "Volume": "volume",
})

if DIAG_START is not None:
    df_diag = df_diag[df_diag["date"] >= pd.to_datetime(DIAG_START)]
if DIAG_END is not None:
    df_diag = df_diag[df_diag["date"] <= pd.to_datetime(DIAG_END)]
df_diag = df_diag.reset_index(drop=True)

if len(df_diag) < 120:
    raise ValueError(f"Need at least ~120 bars for useful DT/DB diagnostics, got {len(df_diag)}")

print(f"[diag] bars={len(df_diag)}")

# Pipeline subset
_df = add_indicators(df_diag)
sig = detect_significant_swings(
    _df,
    left=globals().get("SIG_LEFT", 3),
    right=globals().get("SIG_RIGHT", 3),
    prom_atr=globals().get("SIG_PROM_ATR", 1.5),
    depart_atr=globals().get("SIG_DEPART_ATR", 2.5),
    depart_lookahead=globals().get("SIG_DEPART_LOOK", 10),
    min_swing_sep=globals().get("SIG_MIN_SEP", 7),
)
base_raw = detect_daily_base_candidates(_df, sig, min_base_duration=20, max_retrace=0.50, contraction_tail=5)
base_merged = merge_overlapping_bases(base_raw, _df, contraction_tail=5)
sig = ensure_base_lows_are_significant_swings(_df, sig, base_merged)

dt_a, dt_e, db_a, db_e, dt_db_stats = detect_double_top_bottom_alert_entry(
    _df,
    base_merged,
    approach_atr_mult=DIAG_APPROACH_ATR_MULT,
    min_bar_spacing=DIAG_MIN_BAR_SPACING,
    departure_atr_mult=DIAG_DEPARTURE_ATR_MULT,
)

print("\n[diag] ===== RESULTS =====")
print(f"sig_swings={len(sig)}")
print(f"base_raw={len(base_raw)}  base_merged={len(base_merged)}")
print(f"double_top alert/entry = {len(dt_a)} / {len(dt_e)}")
print(f"double_bottom alert/entry = {len(db_a)} / {len(db_e)}")
print("\n[diag] DT/DB gating stats:")
for k in [
    "dt_approached", "dt_blocked_spacing", "dt_blocked_departure", "dt_emitted",
    "db_approached", "db_blocked_spacing", "db_blocked_departure", "db_emitted",
]:
    print(f"  {k}: {dt_db_stats.get(k, 0)}")

# Quick interpretability hints
if len(base_merged) == 0:
    print("\n[hint] No merged bases detected -> DT/DB cannot trigger. Check base rules first.")
else:
    if dt_db_stats.get("dt_approached", 0) == 0 and dt_db_stats.get("db_approached", 0) == 0:
        print("\n[hint] Price never approached base levels under current approach_atr_mult.")
    if dt_db_stats.get("dt_blocked_spacing", 0) > dt_db_stats.get("dt_emitted", 0):
        print("[hint] DT mostly blocked by spacing. Consider lowering min_bar_spacing for tests.")
    if dt_db_stats.get("dt_blocked_departure", 0) > dt_db_stats.get("dt_emitted", 0):
        print("[hint] DT mostly blocked by departure re-arm. Consider lowering departure_atr_mult for tests.")
    if dt_db_stats.get("db_blocked_spacing", 0) > dt_db_stats.get("db_emitted", 0):
        print("[hint] DB mostly blocked by spacing. Consider lowering min_bar_spacing for tests.")
    if dt_db_stats.get("db_blocked_departure", 0) > dt_db_stats.get("db_emitted", 0):
        print("[hint] DB mostly blocked by departure re-arm. Consider lowering departure_atr_mult for tests.")

# Print a few sample signals if present
if dt_a:
    print("\n[diag] sample DT alert dates:", [str(s.date)[:10] for s in dt_a[:5]])
if db_a:
    print("[diag] sample DB alert dates:", [str(s.date)[:10] for s in db_a[:5]])

In [ ]:
# --- Corrected BASE/DTDB export (full-context detection, then window assignment) ---
# Use this cell if per-window detection misses DT/DB due to lookback/lookforward boundaries.

CTX_TICKERS = globals().get("REVIEW_TICKERS", ["PLTR"])
CTX_YEARS = 5
CTX_MAX_WINDOWS = None
CTX_MAX_PER_WINDOW = 80
CTX_EXPORT_BASE_REGION = True
CTX_APPEND_TO_MAIN_MANIFEST = True

if "_export_plotly_figure" not in globals() or "_build_base_chart" not in globals() or "_build_dt_db_chart" not in globals():
    raise RuntimeError("Run the previous focused export cell first (it defines chart/export helpers).")

ctx_manifest = []
ctx_skipped = []
out_root = Path("../artifacts/setup_review") / RULE_VERSION
out_root.mkdir(parents=True, exist_ok=True)

for ti, ticker in enumerate(CTX_TICKERS):
    print(f"\n[ctx {ti+1}/{len(CTX_TICKERS)}] {ticker}", flush=True)
    try:
        raw_t = yf.Ticker(ticker).history(period=f"{CTX_YEARS}y", interval="1d")
        if raw_t.empty:
            ctx_skipped.append(f"{ticker}: no data")
            print("  SKIP: no data", flush=True)
            continue
        if isinstance(raw_t.columns, pd.MultiIndex):
            raw_t.columns = raw_t.columns.get_level_values(0)
        raw_t = raw_t[["Open", "High", "Low", "Close", "Volume"]].dropna().reset_index()
        raw_t.rename(columns={"Date": "date", "Datetime": "date"}, inplace=True)
        df_t = raw_t.rename(columns={"Open": "open", "High": "high", "Low": "low", "Close": "close", "Volume": "volume"})
    except Exception as e:
        ctx_skipped.append(f"{ticker}: fetch error - {e}")
        print(f"  SKIP fetch error: {e}", flush=True)
        continue

    # 1) Run all structure detection ON FULL CONTEXT FIRST
    dfi = add_indicators(df_t)
    sig = detect_significant_swings(
        dfi,
        left=globals().get("SIG_LEFT", 3),
        right=globals().get("SIG_RIGHT", 3),
        prom_atr=globals().get("SIG_PROM_ATR", 1.5),
        depart_atr=globals().get("SIG_DEPART_ATR", 2.5),
        depart_lookahead=globals().get("SIG_DEPART_LOOK", 10),
        min_swing_sep=globals().get("SIG_MIN_SEP", 7),
    )
    base_raw = detect_daily_base_candidates(dfi, sig, min_base_duration=20, max_retrace=0.50, contraction_tail=5)
    base_merged = merge_overlapping_bases(base_raw, dfi, contraction_tail=5)
    sig = ensure_base_lows_are_significant_swings(dfi, sig, base_merged)
    dt_a, dt_e, db_a, db_e, dt_db_stats = detect_double_top_bottom_alert_entry(
        dfi,
        base_merged,
        approach_atr_mult=0.50,
        min_bar_spacing=7,
        departure_atr_mult=1.0,
    )

    print(f"  full-context counts: bases={len(base_merged)} dt={len(dt_a)} db={len(db_a)}", flush=True)
    print(f"  gating: dt_approached={dt_db_stats.get('dt_approached',0)} dt_emitted={dt_db_stats.get('dt_emitted',0)} db_approached={dt_db_stats.get('db_approached',0)} db_emitted={dt_db_stats.get('db_emitted',0)}", flush=True)

    # 2) Assign precomputed full-context signals into half-year windows
    windows = generate_half_year_windows(df_t, years=CTX_YEARS)
    if CTX_MAX_WINDOWS is not None:
        windows = windows[:CTX_MAX_WINDOWS]
    print(f"  windows={len(windows)}", flush=True)

    # enrich with date keys for efficient window filtering
    base_with_dates = []
    for i, b in enumerate(base_merged):
        end_idx = int(b["end_idx"])
        base_with_dates.append((i, b, pd.to_datetime(dfi["date"].iloc[end_idx])))

    dt_with_dates = [(i, s, pd.to_datetime(s.date)) for i, s in enumerate(dt_a)]
    db_with_dates = [(i, s, pd.to_datetime(s.date)) for i, s in enumerate(db_a)]

    for wi, (w_start, w_end) in enumerate(windows):
        ws = pd.to_datetime(w_start)
        we = pd.to_datetime(w_end)
        print(f"    window {wi+1}/{len(windows)} [{str(ws)[:10]} to {str(we)[:10]}]", flush=True)

        # choose signals by alert/end date inside window
        w_bases = [(i, b) for (i, b, d) in base_with_dates if ws <= d <= we]
        w_dt = [(i, s) for (i, s, d) in dt_with_dates if ws <= d <= we]
        w_db = [(i, s) for (i, s, d) in db_with_dates if ws <= d <= we]

        print(f"      in-window: bases={len(w_bases)} dt={len(w_dt)} db={len(w_db)}", flush=True)

        if CTX_EXPORT_BASE_REGION:
            for bi, (global_i, b) in enumerate(w_bases[:CTX_MAX_PER_WINDOW]):
                date_tag = str(dfi["date"].iloc[int(b["end_idx"])])[:10]
                chart_id = f"{ticker}_BASE_REGION_{date_tag}_{global_i}"
                out_dir = out_root / ticker / "BASE_REGION"
                out_dir.mkdir(parents=True, exist_ok=True)
                out_png = out_dir / f"{date_tag}_{global_i}.png"
                rel_path = _export_plotly_figure(_build_base_chart(dfi, ticker, b), out_png)
                ctx_manifest.append({
                    "chart_id": chart_id,
                    "ticker": ticker,
                    "setup_type": "BASE_REGION",
                    "alert_date": date_tag,
                    "alert_price": round(float(b.get("base_high", np.nan)), 2) if np.isfinite(float(b.get("base_high", np.nan))) else None,
                    "window_start": str(ws)[:10],
                    "window_end": str(we)[:10],
                    "rule_version": RULE_VERSION,
                    "chart_path": rel_path,
                    "direction": "BOTH",
                })

        for _, (global_i, s) in enumerate(w_dt[:CTX_MAX_PER_WINDOW]):
            date_tag = str(s.date)[:10]
            chart_id = f"{ticker}_DOUBLE_TOP_{date_tag}_{global_i}"
            out_dir = out_root / ticker / "DOUBLE_TOP"
            out_dir.mkdir(parents=True, exist_ok=True)
            out_png = out_dir / f"{date_tag}_{global_i}.png"
            rel_path = _export_plotly_figure(_build_dt_db_chart(dfi, ticker, s, "DOUBLE_TOP"), out_png)
            ctx_manifest.append({
                "chart_id": chart_id,
                "ticker": ticker,
                "setup_type": "DOUBLE_TOP",
                "alert_date": date_tag,
                "alert_price": round(float(s.price), 2),
                "window_start": str(ws)[:10],
                "window_end": str(we)[:10],
                "rule_version": RULE_VERSION,
                "chart_path": rel_path,
                "direction": "SHORT",
            })

        for _, (global_i, s) in enumerate(w_db[:CTX_MAX_PER_WINDOW]):
            date_tag = str(s.date)[:10]
            chart_id = f"{ticker}_DOUBLE_BOTTOM_{date_tag}_{global_i}"
            out_dir = out_root / ticker / "DOUBLE_BOTTOM"
            out_dir.mkdir(parents=True, exist_ok=True)
            out_png = out_dir / f"{date_tag}_{global_i}.png"
            rel_path = _export_plotly_figure(_build_dt_db_chart(dfi, ticker, s, "DOUBLE_BOTTOM"), out_png)
            ctx_manifest.append({
                "chart_id": chart_id,
                "ticker": ticker,
                "setup_type": "DOUBLE_BOTTOM",
                "alert_date": date_tag,
                "alert_price": round(float(s.price), 2),
                "window_start": str(ws)[:10],
                "window_end": str(we)[:10],
                "rule_version": RULE_VERSION,
                "chart_path": rel_path,
                "direction": "LONG",
            })

ctx_manifest_path = out_root / "manifest_base_dt_db_context_first.json"
with open(ctx_manifest_path, "w") as f:
    json.dump(ctx_manifest, f, indent=2, default=str)

if CTX_APPEND_TO_MAIN_MANIFEST and ctx_manifest:
    main_manifest_path = out_root / "manifest.json"
    existing = []
    if main_manifest_path.exists():
        with open(main_manifest_path) as f:
            existing = json.load(f)
    existing_ids = {x.get("chart_id") for x in existing}
    merged = existing + [x for x in ctx_manifest if x.get("chart_id") not in existing_ids]
    with open(main_manifest_path, "w") as f:
        json.dump(merged, f, indent=2, default=str)
    print(f"Appended {len(merged)-len(existing)} context-first records into manifest.json", flush=True)

if ctx_skipped:
    with open(out_root / "skipped_base_dt_db_context_first.log", "w") as f:
        f.write("\n".join(ctx_skipped))

from collections import Counter
cc = Counter([x["setup_type"] for x in ctx_manifest])
print("\n[context-first done]")
print(f"manifest: {ctx_manifest_path.resolve()}")
print(f"counts: {dict(cc)}")
print(f"skipped: {len(ctx_skipped)}")